# 1. Cleaning EDA

## Overview

This notebook loads Discogs' raw masters and releases dumps, cleans them, and combines them into a single `works_df`: one row per musical work, ready for feature engineering and snapshot generation in later stages of the project.

The cleaning pipeline runs as a sequence of filters, each narrowing `works_df` further:

- **Exclusion filters**: drop non-music, Classical, Brass & Military and placeholder-artist masters (Children's, test/dummy records, Entirely Incorrect data quality), and drop compilation/unofficial releases.
- **Combine masters with synthetic standalone works**: recover artists with 3+ standalone releases but no Discogs Master, applying the same exclusion rules to them.
- **Master missing-year recovery**: fill a Master's missing year from the earliest year of its own Releases, before dedup.
- **Exact-match dedup**: collapse exact duplicates (same artist, normalized title, year).
- **Reissue/remaster removal**: drop later re-releases of an earlier work by the same artist.
- **Title-pattern exclusions**: remove Live, Greatest Hits/best-of/anthology, Remixes, Tribute, (Acoustic)/(Instrumental), Unplugged, and radio-edit/BBC-session titles, each a repackaging of existing material rather than new creative work.
- **Fuzzy-dedup**: collapse wording-level near-duplicate titles within the same artist (guarded against numbered/episodic series false positives).
- **Catalog-number dedup**: collapse same-artist/same-title groups that share a real label catalog number but carry no reissue tag.
- **Mass-upload artist exclusion**: drop artist catalogs that read as bulk/algorithmic uploads rather than a genuine discography.

The final `works_df` has 5,949,217 rows and 9 columns, checkpointed to `works_checkpoint.parquet`.

The reasoning and numbers for each step are in `docs/decisions_core.md`.

## Imports

In [1]:
# Make src/ importable from inside notebooks/, so we can reuse the
# already-validated parsing code instead of rewriting it here.
import sys
sys.path.append('../src')

import pandas as pd
from parse_dumps import convert_masters
import re



## 1. Masters structure check

### 1. Check structure - Masters

In [2]:
# Convert only the first 1,000 Masters from the dump
# into a small Parquet file. limit=1000 keeps this fast and safe
# to inspect manually, while still reading actual production data.
summary = convert_masters(
    xml_path="../data/raw/discogs_20260901_masters.xml",
    out_path="../data/processed/masters_sample.parquet",
    limit=1000
)

# summary tells us how many rows were written and how many records
# failed to parse (if any), our first sanity check on the real data.
summary

Done: 1,000 rows -> ../data/processed/masters_sample.parquet (0.0s, 0 parse errors)


{'tag': 'master',
 'xml_path': '../data/raw/discogs_20260901_masters.xml',
 'output_path': '../data/processed/masters_sample.parquet',
 'rows_written': 1000,
 'parse_errors': 0,
 'elapsed_seconds': 0.0}

In [3]:
# Load the small sample we just wrote and look at the first rows.
# This confirms the real XML parses into the same flat structure
# (master_id, artist_ids, title, year, genres, styles, etc.)
# we already validated against the demo sample.
df = pd.read_parquet("../data/processed/masters_sample.parquet")
df

,master_id,main_release_id,artist_ids,artist_names,title,year,genres,styles,data_quality
0,113,116925,[3225],[Vince Watson],Moments In Time,2002,[Electronic],"[Techno, Tech House]",Correct
1,114,42056,[3225],[Vince Watson],Biologique,1999,[Electronic],"[Deep House, Tech House, Techno]",Correct
2,115,336873,[233125],[Soleil (2)],Soleil,2004,[Electronic],"[IDM, Electro]",Correct
3,116,5415,[1162],[Blue Arsed Fly],Knackered EP,1997,[Electronic],[Techno],Correct
4,117,91899,[5508],[Origin],Wide-Eyed Angel,1999,[Electronic],[Progressive Trance],Correct
...,...,...,...,...,...,...,...,...,...
995,1149,56456,[7385],[Phuture Assassins],Roots 'N Future,1993,[Electronic],"[Breakbeat, Hardcore, Jungle]",Correct
996,1150,1338116,"[15147, 15204]","[Endymion, Evil Activities]",To Claim The Future,2008,[Electronic],"[Hardcore, Gabber]",Correct
997,1151,842430,[303434],[Solar Movement],Under The Sun / Close Your Eyes,2006,[Electronic],[Trance],Needs Vote
998,1152,1530923,[327234],[Limewax],The Kristall Weizen EP,2008,[Electronic],[Drum n Bass],Correct


### Interpretation

The 1,000-row sample from the real Masters dump has the same flat structure as the demo sample:

- Columns match expectations: `master_id`, `main_release_id`,
  `artist_ids`, `artist_names`, `title`, `year`, `genres`, `styles`,
  `data_quality`, all present and correctly typed (lists for
  multi-valued fields, int for year/ids).
- Multi-artist collaborations parse correctly: e.g. master_id 1150
  ("To Claim The Future") has two artist_ids/artist_names in sync
  (`[15147, 15204]` / `[Endymion, Evil Activities]`).
- Styles are consistently multi-valued: most Masters carry 1-4
  styles under a single genre (almost all "Electronic" in this sample,
  expected since master_id ranges are roughly chronological/sequential
  in Discogs, not randomly distributed across genres).
- `data_quality` is not always "Correct": master_id 1151 shows
  "Needs Vote", confirming this field has real variation worth
  checking across the full dataset before deciding whether/how to
  filter on it.
- No missing/null values observed in this sample across any column.

### Decision

The real XML parses cleanly with `convert_masters()` with no structural
surprises vs. the demo sample. Safe to proceed to a larger-scale
conversion (or targeted full-field null/cardinality checks) before
moving to Releases.

## 2. Releases structure check

### 2. Check structure - Releases

In [4]:
# Convert only the first 1,000 Releases from the dump
# into a small Parquet file. limit=1000 keeps this fast and safe
# to inspect manually, while still reading actual production data.
from parse_dumps import convert_releases

summary = convert_releases(
    xml_path="../data/raw/discogs_20260901_releases.xml",
    out_path="../data/processed/releases_sample.parquet",
    limit=1000
)

# summary tells us how many rows were written and how many records
# failed to parse (if any), our first sanity check on the real data.
summary

Done: 1,000 rows -> ../data/processed/releases_sample.parquet (0.1s, 0 parse errors)


{'tag': 'release',
 'xml_path': '../data/raw/discogs_20260901_releases.xml',
 'output_path': '../data/processed/releases_sample.parquet',
 'rows_written': 1000,
 'parse_errors': 0,
 'elapsed_seconds': 0.1}

In [5]:
# Load the small sample we just wrote and look at the first rows.
# Releases carry extra fields Masters don't (country, formats,
# tracklist, barcodes, labels, etc.) since they describe a specific
# physical pressing rather than the abstract work.
df_releases = pd.read_parquet("../data/processed/releases_sample.parquet")
df_releases

,release_id,status,artist_ids,artist_names,title,master_id,is_main_release,country,released,genres,...,formats,format_descriptions,data_quality,tracklist,n_tracks,barcodes,matrix_numbers,labels,format_qty,format_text
0,1,NaN,[1],[The Persuader],Stockholm,1660109,True,Sweden,1999-03-00,[Electronic],...,[Vinyl],"[12"", 33 ⅓ RPM]",Correct,"[{'position': 'A', 'title': 'Östermalm', 'dura...",6,[],[MPO SK 032 A1 G PHRUPMASTERGENERAL T2T LONDON...,"[{'name': 'Svek', 'catno': 'SK032'}]",[2],[]
1,2,NaN,[2],[Mr. James Barth & A.D.],Knockin' Boots (Vol 2 Of 2),713738,True,Sweden,1998-06-00,[Electronic],...,[Vinyl],"[12"", 33 ⅓ RPM]",Correct,"[{'position': 'A1', 'title': 'A Sea Apart', 'd...",4,[],"[MPO SK026-A -J.T.S.-, MPO SK026-B -J.T.S.-]","[{'name': 'Svek', 'catno': 'SK 026'}, {'name':...",[1],[]
2,3,NaN,[3],[Josh Wink],Profound Sounds Vol. 1,66526,False,US,1999-07-13,[Electronic],...,[CD],"[Compilation, Mixed]",Correct,"[{'position': '1', 'title': 'D2', 'duration': ...",14,"[0 7464-63628-2 2, 074646362822]",[CPDP-098103 G4 1A 01],"[{'name': 'Ruffhouse Records', 'catno': 'CK 63...",[1],[]
3,4,NaN,[21],[Faze Action],Moving Cities,104980,False,US,1999-11-02,[Electronic],...,[CD],[Album],Needs Vote,"[{'position': '1', 'title': 'Moving Cities', '...",12,"[0 9362-47555-2 4, 093624755524]",[[wea mfg. OLYPHANT logo] X9694 1 47555-2 RE-1...,"[{'name': 'F-111 Records', 'catno': '9 47555-2...",[1],[]
4,5,NaN,[22],[Datacide],Flowerhead,169944,True,Germany,1995-01-15,[Electronic],...,[CD],"[Album, Limited Edition]",Correct,"[{'position': '1', 'title': 'Flashback Signal'...",5,[],[CD PLANT MFG RI 026 CDM01],"[{'name': 'Rather Interesting', 'catno': 'RI 0...",[1],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1027,NaN,[15581],[100 Hz],EP 2,235899,True,UK,1996-00-00,[Electronic],...,[Vinyl],"[12"", EP, 33 ⅓ RPM]",Correct,"[{'position': 'A', 'title': 'Warp', 'duration'...",3,[],[],"[{'name': 'Pacific Records', 'catno': 'fic009'}]",[1],[None]
996,1028,NaN,[1475],[DJ du Jour],Futon Funk EP,0,False,UK,1999-00-00,[Electronic],...,[Vinyl],"[12"", EP]",Complete and Correct,"[{'position': 'A1', 'title': 'Nothing Kinky', ...",4,[],[],"[{'name': 'Neon Heights', 'catno': 'NH 009'}]",[1],[None]
997,1029,NaN,[1475],[DJ du Jour],Pussy Galore EP,1945675,True,UK,1998,[Electronic],...,[Vinyl],"[12"", EP]",Needs Vote,"[{'position': 'A', 'title': 'Gene Filter', 'du...",3,[],[],"[{'name': 'Neon Heights', 'catno': 'NH 007'}]",[1],[]
998,1030,NaN,[1476],[Metro Area],Metro Area 2,11076,True,US,2000,[Electronic],...,[Vinyl],"[12"", 45 RPM, 33 ⅓ RPM]",Needs Vote,"[{'position': 'A', 'title': 'The Art Of Hot', ...",3,[],[],"[{'name': 'Environ', 'catno': 'ENV 010'}]",[1],[]


### Interpretation

The 1,000-row sample from the real Releases dump introduces fields
Masters don't have, since a Release describes one specific pressing
rather than the abstract work:

- **New columns**: `release_id`, `status`, `master_id`,
  `is_main_release`, `country`, `released`, `formats`,
  `format_descriptions`, `format_qty`, `format_text`, `tracklist`,
  `n_tracks`, `barcodes`, `matrix_numbers`, `labels`, all present and
  correctly typed (lists for multi-valued fields, struct lists for
  `tracklist`/`labels`).
- **`status` is entirely `NaN`** in this sample. Checked the raw XML
  directly: `<release id="1">` carries no `status` attribute at all in
  this dump, so this is a real data characteristic, not a parsing bug.
- **`data_quality` is far less "vetted" than on Masters**: `Needs Vote`
  627, `Correct` 341, `Complete and Correct` 31, `Needs Minor Changes`
  1 (the opposite skew from the Masters sample, where `Correct`
  dominated).
- **Pressing-level detail is present and usable**: tracklist, barcodes
  and matrix numbers all populate as expected, which matters for
  later duplicate/reissue resolution (EDA cleaning decision #9).

### Decision

The real Releases XML parses cleanly with `convert_releases()`. Given
the `status` null rate and the `data_quality` skew differ sharply from
Masters, treat the two tables' quality fields separately rather than
applying one shared threshold, safe to proceed to a larger-scale
conversion before finalizing any filters.

## 3. Loading and first-pass summary

### Parquet files

Note: Parquet files already generated

The Masters and Releases Parquet files used from this point on were
already generated earlier in this project's development, using the
same `convert_masters()` / `convert_releases()` functions in
`src/parse_dumps.py`, with no code changes since.

Structure and correctness were validated on both a demo sample and a
real-data sample (see cells above) before the full conversion was run.
The parser is deterministic, so re-running it here would reproduce the
same output at the cost of several hours, which is why the files below
are used directly instead.

For reproducibility, they can be regenerated at any time with:

```bash
python -m src.parse_dumps masters data/raw/discogs_20260901_masters.xml data/processed/masters.parquet
python -m src.parse_dumps releases data/raw/discogs_20260901_releases.xml data/processed/releases.parquet
```

**Files used from here on:**
- `data/processed/masters.parquet` (2,589,349 rows)
- `data/processed/releases.parquet` (19,417,067 rows)

In [6]:
# Load both full real-data tables for the column-by-column pass.
masters_df = pd.read_parquet("../data/processed/masters.parquet")
releases_df = pd.read_parquet("../data/processed/releases.parquet")

print("masters_df shape:", masters_df.shape)
print("releases_df shape:", releases_df.shape)

masters_df shape: (2589349, 9)
releases_df shape: (19417067, 21)


In [7]:
print("--- masters_df dtypes ---")
print(masters_df.dtypes)
print()
print("--- masters_df null counts ---")
print(masters_df.isnull().sum())

--- masters_df dtypes ---
master_id           int64
main_release_id     int64
artist_ids         object
artist_names       object
title                 str
year                int32
genres             object
styles             object
data_quality          str
dtype: object

--- masters_df null counts ---


master_id          0
main_release_id    0
artist_ids         0
artist_names       0
title              0
year               0
genres             0
styles             0
data_quality       0
dtype: int64


In [8]:
print("--- releases_df dtypes ---")
print(releases_df.dtypes)
print()
print("--- releases_df null counts ---")
print(releases_df.isnull().sum())

--- releases_df dtypes ---
release_id              int64
status                    str
artist_ids             object
artist_names           object
title                     str
master_id               int64
is_main_release          bool
country                   str
released                  str
genres                 object
styles                 object
formats                object
format_descriptions    object
data_quality              str
tracklist              object
n_tracks                int32
barcodes               object
matrix_numbers         object
labels                 object
format_qty             object
format_text            object
dtype: object

--- releases_df null counts ---


release_id                    0
status                 19417067
artist_ids                    0
artist_names                  0
title                         0
master_id                     0
is_main_release               0
country                  608023
released                2584225
genres                        0
styles                        0
formats                       0
format_descriptions           0
data_quality                  0
tracklist                     0
n_tracks                      0
barcodes                      0
matrix_numbers                0
labels                        0
format_qty                    0
format_text                   0
dtype: int64


### Summary

`masters_df` has zero nulls across all 9 columns, no cleaning needed there.

`releases_df` has nulls in three columns:
- `status`: null in all 19,417,067 rows. Checked the raw XML directly (`<release id="1">`,
  no `status` attribute present), confirming this field does not exist in this dump version.
  It carries no information and is dropped below.
- `country`: 608,023 nulls (~3%), plausible for releases where Discogs never recorded one.
- `released`: 2,584,225 nulls (~13%), consistent with releases that have no date on file.

### Decision

Drop `status` from `releases_df`. Keep `country` and `released` as is for now, their nulls
are expected and will be handled later depending on how they're used.

In [9]:
releases_df = releases_df.drop(columns=["status"])

## 4. Cleaning masters_df column by column

### Checking the structure of the data

### Question: is master_id a unique key in masters_df?

In [10]:
print("masters_df rows:", len(masters_df))
print("unique master_id:", masters_df["master_id"].nunique())
print("duplicated master_id rows:", masters_df["master_id"].duplicated().sum())

masters_df rows: 2589349
unique master_id: 2589349
duplicated master_id rows: 0


In [11]:
print("releases_df rows:", len(releases_df))
print("releases with master_id == 0:", (releases_df["master_id"] == 0).sum())
print("releases with master_id != 0:", (releases_df["master_id"] != 0).sum())

releases_df rows: 19417067


releases with master_id == 0: 8076985
releases with master_id != 0: 11340082


#### Interpretation: master_id uniqueness and coverage

`masters_df["master_id"]` is a unique key: 2,589,349 rows, 2,589,349 unique values, 0 duplicates.
Safe to treat as the primary key for Masters going forward.

`releases_df["master_id"]` tells a different story: 8,076,985 releases (41.6%) have
master_id == 0, meaning they are not grouped under any Master at all. The remaining
11,340,082 releases (58.4%) do belong to a Master.

#### Decision

Confirms the planned work_id logic: work_id = master_id when master_id != 0, otherwise a
synthetic id is needed. Nearly 42% of releases will require that synthetic path, this is
too large a share to treat as an edge case, it is handled at the work-building stage.

### Question: how clean is the year column in masters_df?

In [12]:
print("masters_df rows:", len(masters_df))
print("year == 0:", (masters_df["year"] == 0).sum())
print("year min:", masters_df["year"].min())
print("year max:", masters_df["year"].max())
print()
print(masters_df["year"].describe())

masters_df rows: 2589349
year == 0: 173369
year min: 0
year max: 2026

count    2.589349e+06
mean     1.862010e+03
std      4.991812e+02
min      0.000000e+00
25%      1.977000e+03
50%      1.997000e+03
75%      2.012000e+03
max      2.026000e+03
Name: year, dtype: float64


In [13]:
masters_df.loc[masters_df["year"] == 0, "year"] = pd.NA

print("missing year after fix:", masters_df["year"].isna().sum())

missing year after fix: 173369


#### Interpretation

173,369 rows (6.7%) have year == 0, Discogs' way of marking "no year recorded,"
not an actual date. Max is 2026, consistent with current releases, no corrupted
future-year values. Min of 0 confirms 0 is used as a sentinel for missing rather
than NaN, which also explains why the mean (1862) looks implausibly low, it's
being dragged down by hundreds of thousands of 0s mixed into a real average.

#### Decision

Treat year == 0 as missing. Convert those to NaN (or pd.NA) before computing any
further statistics or using year in modeling, otherwise every mean/distribution
built on this column will stay distorted the same way. The 6.7% missing rate
itself is small enough not to drop those rows outright, this should be a decision
for the work-building stage depending on whether year is required there.

### Question: artist_names cardinality and compilation-style entries

In [14]:
# artist_names stores a list of artist names per Master (some Masters have
# multiple artists credited). Lists are unhashable, so nunique() and
# value_counts() fail directly on this column.
#
# Join each row's list into a single comma-separated string first, this
# gives us a hashable value we can count, and it also matches how
# multi-artist credits will typically be displayed anyway.
artist_names_str = masters_df["artist_names"].apply(lambda names: ", ".join(names))

print("unique artist_names values:", artist_names_str.nunique())
print()

# The most frequent values are where compilation-style entries
# (e.g. "Various", "Various Artists") tend to surface, since they are
# shared across a huge number of unrelated Masters, unlike a real
# single-artist name.
print(artist_names_str.value_counts().head(20))

unique artist_names values: 927468

artist_names
Various                    165403
Unknown Artist               6711
No Artist                    1885
Elvis Presley                1872
The Beatles                  1509
The Rolling Stones           1258
Frank Sinatra                1160
Louis Armstrong               783
Iron Maiden                   779
David Bowie                   731
Metallica                     728
Nat King Cole                 717
Johnny Cash                   703
Wolfgang Amadeus Mozart       691
Bing Crosby                   675
Johnny Hallyday               659
Ray Charles                   654
Bob Dylan                     637
Ella Fitzgerald               609
Pink Floyd                    605
Name: count, dtype: int64


#### Interpretation

927,468 unique artist_names strings across 2,589,349 masters, a highly long-tailed
distribution, as expected for an artist dataset.

Three non-artist placeholder values stand out at the top:
- "Various": 165,403 masters (6.4%), compilation releases credited to multiple
  unrelated artists.
- "Unknown Artist": 6,711 masters.
- "No Artist": 1,885 masters.

Together these account for ~174,000 masters (6.7%) that do not represent a real
artist's work and should not be used for next-work prediction, since there is no
single artist to predict a "next work" for.

The rest of the top 20 are real, highly prolific artists (Elvis Presley, The Beatles,
etc.), appearing frequently because Discogs tracks reissues and compilations as
separate masters, not because of any data problem.

#### Decision

Exclude masters where artist_names is "Various", "Unknown Artist", or "No Artist"
from the Works dataset. This confirms and operationalizes the already-decided
"Various artists" exclusion rule, extending it to the two other non-artist
placeholders found here.

In [15]:
# Case-insensitive now: an earlier run of this cell did an exact,
# case-sensitive match, which let through case variants like "various"
# and "VARIOUS" (found later, 2026-10-06, while reviewing works_df titles).
# Comparing on a lowercased copy catches those too, without changing the
# original artist_names_str values used elsewhere.
NON_ARTIST_PLACEHOLDERS = {"various", "unknown artist", "no artist"}

is_non_artist = artist_names_str.str.lower().isin(NON_ARTIST_PLACEHOLDERS)
print("masters excluded as non-artist:", is_non_artist.sum())

masters_df = masters_df[~is_non_artist].reset_index(drop=True)
print("masters_df shape after exclusion:", masters_df.shape)

masters excluded as non-artist: 174049


masters_df shape after exclusion: (2415300, 9)


### Question: genres and styles distribution in masters_df, where do Classical and Brass & Military exclusions apply?

In [16]:
# genres and styles also store a list per Master (a Master can belong to more
# than one genre/style). Unlike artist_names, here we want to count each
# individual genre/style across the dataset, not unique combinations, so we
# explode the list into one row per genre instead of joining it into a string.
genres_exploded = masters_df["genres"].explode()
styles_exploded = masters_df["styles"].explode()

print("--- genres value_counts ---")
print(genres_exploded.value_counts())
print()
print("--- styles value_counts (top 20) ---")
print(styles_exploded.value_counts().head(20))

--- genres value_counts ---
genres
Rock                      748131
Electronic                647555
Pop                       485420
Folk, World, & Country    313366
Jazz                      190037
Funk / Soul               160223
Classical                 151884
Hip Hop                   113954
Latin                     104105
Stage & Screen             65466
Reggae                     58243
Blues                      48553
Non-Music                  38685
Children's                 22390
Brass & Military            7988
1                             10
Name: count, dtype: int64

--- styles value_counts (top 20) ---
styles
Pop Rock            90904
House               90536
Vocal               89700
Punk                86070
Experimental        83772
Techno              70352
Ambient             65822
Hardcore            64547
Indie Rock          63619
Alternative Rock    62805
Folk                59553
Synth-pop           57114
Soul                55166
Country             54353
Ba

#### Question: what is the "1" genre anomaly, and what does Non-Music actually contain?

In [17]:
# Look at the 10 rows with the "1" genre anomaly before deciding what to do with them.
anomaly_mask = masters_df["genres"].apply(lambda g: "1" in g)
print(masters_df.loc[anomaly_mask, ["master_id", "title", "artist_names", "genres", "year"]])

         master_id                                        title  \
27918        30486                          Go Away Little Girl   
140177      153959  Vietnam / You Can Get It If You Really Want   
277128      303330                       * [Without Vocabulary]   
417224      455639                                         Love   
1071526    1179717                                 Wildstyle EP   

          artist_names              genres    year  
27918     [Easy Going]  [1, 1, Electronic]  1982.0  
140177   [Jimmy Cliff]        [1, 1, Rock]     NaN  
277128    [David Jack]  [1, 1, Electronic]  2003.0  
417224        [Sirene]  [1, 1, Electronic]  2001.0  
1071526        [Erase]  [1, 1, Electronic]  2001.0  


#### Interpretation: the "1" genre anomaly

The genre value_counts showed "1" appearing 10 times, which is not a real
Discogs genre category (the real list is Rock, Electronic, Pop, Classical,
etc.), so this looked like a data problem and was inspected directly rather
than assumed.

Only 5 masters are actually affected, each one has "1" appearing twice inside
its genres list, alongside one real, valid genre (Electronic in 4 cases, Rock
in 1). That duplication is why the exploded count showed 10 instead of 5.

#### Decision

Strip "1" out of the genres list wherever it appears, keeping the real genre
intact. Only the "1" artifact is removed, the row itself is kept, since each
affected master still has a legitimate genre tag and dropping the row would
lose valid data for no reason. The tag still appears on 8 Works of the final table and is excluded from genre analyses and features (see `docs/decisions_core.md`).

In [18]:
# Remove the "1" artifact from genres, keeping any real genres in the same row.
masters_df["genres"] = masters_df["genres"].apply(lambda g: [x for x in g if x != "1"])

#### Question: what does the Non-Music genre actually contain?

In [19]:
# Inspect a sample of masters tagged Non-Music to understand what's actually in there.
non_music_mask = masters_df["genres"].apply(lambda g: "Non-Music" in g)
print("masters tagged Non-Music:", non_music_mask.sum())
print()
print(masters_df.loc[non_music_mask, ["master_id", "title", "artist_names", "genres", "styles"]].head(20))

masters tagged Non-Music: 38685



      master_id                                              title  \
431         583  Eroismo E Pagliaccismo - Rarità E Inediti 1997...   
551         710                          10.000 Chickens' Symphony   
820        1007                             The Lady Or The Tiger?   
1553       1808  A Single Chunk Of Preliminary Drama And The Tr...   
3076       3456                                          Evolution   
3275       3681                                        Man Vermist   
3287       3693                         Tunnel Zonder Vluchtstrook   
3786       4227                                    They Don't Know   
3992       4444                               Back And Forth 06Six   
4137       4600                                         Binnensuis   
4223       4696                                          Watershed   
5102       5664                                    Can You Feel It   
5221       5796  UHF (Original Motion Picture Soundtrack And Ot...   
5226       5801     

#### Interpretation

The sample above confirms Non-Music masters are spoken word, comedy, and
interview content, not music, regardless of who released them (even
well-known recording artists like "Weird Al" Yankovic or Gil Scott-Heron
have Non-Music masters alongside their real music).

#### Decision

Exclude masters tagged Non-Music from the Works dataset (38,685 masters,
1.5%), since spoken word, comedy, and interview content isn't music, even
when released by a real recording artist.

Also exclude masters tagged Children's (22,390 masters, 0.9%), per the same
reasoning, not relevant to musical style prediction.


In [20]:
EXCLUDED_GENRES_2 = {"Non-Music", "Children's"}

is_excluded_genre_2 = masters_df["genres"].apply(lambda g: any(genre in EXCLUDED_GENRES_2 for genre in g))
print("masters excluded (Non-Music / Children's):", is_excluded_genre_2.sum())

masters_df = masters_df[~is_excluded_genre_2].reset_index(drop=True)
print("masters_df shape after exclusion:", masters_df.shape)

masters excluded (Non-Music / Children's): 55065


masters_df shape after exclusion: (2360235, 9)


#### Question: does the Classical / Brass & Military exclusion already flagged in src/works.py apply here too?

In [21]:
# src/works.py already defines DEFAULT_EXCLUDED_GENRES = {"Classical", "Brass & Military"}
# for the Works table, but that constant isn't backed by an EDA count yet
# and isn't actually used anywhere. Recompute the current counts here,
# after the exclusions already applied above, before using it.
from works import DEFAULT_EXCLUDED_GENRES

for genre in sorted(DEFAULT_EXCLUDED_GENRES):
    count = masters_df["genres"].apply(lambda g: genre in g).sum()
    print(f"masters tagged {genre!r}: {count}")

masters tagged 'Brass & Military': 7737


masters tagged 'Classical': 149308


#### Interpretation

Classical and Brass & Military are both real, well-populated genres in this
data, not anomalies, but [works.py](../src/works.py) already scopes them out of
the Works dataset: Classical is mostly performances of compositions written
long before the performer was credited, often by many different performers,
and Brass & Military is dominated by ensembles and bands rather than a
single artist's own trajectory. Neither fits the WHEN/WHAT questions' core
assumption of one artist with their own evolving, datable body of work.

#### Decision

Exclude masters tagged Classical or Brass & Military from masters_df, using
the existing `DEFAULT_EXCLUDED_GENRES` constant from `src/works.py` rather
than redefining the set here, so there is exactly one place this scope
decision lives.

In [22]:
is_excluded_genre_3 = masters_df["genres"].apply(lambda g: any(genre in DEFAULT_EXCLUDED_GENRES for genre in g))
print("masters excluded (Classical / Brass & Military):", is_excluded_genre_3.sum())

masters_df = masters_df[~is_excluded_genre_3].reset_index(drop=True)
print("masters_df shape after exclusion:", masters_df.shape)

masters excluded (Classical / Brass & Military): 155769
masters_df shape after exclusion: (2204466, 9)


### Question: full styles distribution, how many masters have no style, and are there anomalies like the genres "1" artifact?

In [23]:
styles_exploded = masters_df["styles"].explode()

print("unique styles:", styles_exploded.nunique())
print("total style tags:", len(styles_exploded))
print()

# How many masters have zero styles at all vs one vs multiple
style_counts_per_master = masters_df["styles"].apply(len)
print("masters with 0 styles:", (style_counts_per_master == 0).sum())
print("masters with 1 style:", (style_counts_per_master == 1).sum())
print("masters with 2+ styles:", (style_counts_per_master >= 2).sum())
print()

# Full distribution, not just top 20, to scan for anomalies like the "1" artifact in genres
print(styles_exploded.value_counts().to_string())

unique styles: 718
total style tags: 3653521

masters with 0 styles: 294388
masters with 1 style: 981727
masters with 2+ styles: 928351

styles
House                          90331
Pop Rock                       89881
Punk                           85684
Vocal                          84936
Experimental                   78104
Techno                         70107
Hardcore                       64392
Indie Rock                     63326
Alternative Rock               62276
Ambient                        61952
Synth-pop                      56492
Folk                           55770
Soul                           54982
Country                        53800
Electro                        47928
Ballad                         47488
Disco                          46296
Trance                         45304
Chanson                        43776
Black Metal                    37755
Rock & Roll                    36097
Hard Rock                      35717
Psychedelic Rock               35415
Schla

In [24]:
# Look at the masters with 0 styles to see if they have any genres, and if so, how many.

no_style_mask = masters_df["styles"].apply(len) == 0

print("masters with 0 styles:", no_style_mask.sum())
print("of those, with 0 genres too:", (masters_df.loc[no_style_mask, "genres"].apply(len) == 0).sum())
print("of those, with 1+ genres:", (masters_df.loc[no_style_mask, "genres"].apply(len) >= 1).sum())

masters with 0 styles: 294388
of those, with 0 genres too: 1
of those, with 1+ genres: 294387


#### Interpretation: styles distribution and coverage

718 unique styles across 3,653,521 style tags, no anomalies like the genres
"1" artifact found, every value is a real, recognizable music style, even the
rare long-tail ones (e.g. "Gagaku", "Zarzuela", "Honkyoku") are legitimate
regional/genre styles, not parsing noise.

Coverage is uneven:
- 294,388 masters (13.4%) have 0 styles at all.
- 981,727 (44.5%) have exactly 1 style.
- 928,351 (42.1%) have 2 or more styles.

Checked whether the no-style masters are a broader data gap: of the 294,388
with 0 styles, 294,387 (99.9997%) still have at least one genre. Only 1 master
has neither genre nor style. So missing styles is not a symptom of generally
missing metadata, genre coverage stays essentially complete regardless. This
matters for feature engineering: styles can't be relied on alone, genre
should be the fallback wherever style is empty.

#### Decision

No cleaning action needed on styles itself, it's clean and the coverage gap is
explained. The single master with 0 genres and 0 styles carries no usable
genre/style signal at all and is dropped below, everything else is kept as is.


In [25]:
# Find and inspect the one master with neither genre nor style before dropping it.
no_genre_no_style_mask = (masters_df["genres"].apply(len) == 0) & (masters_df["styles"].apply(len) == 0)
print(masters_df.loc[no_genre_no_style_mask])

        master_id  main_release_id artist_ids artist_names   title  year  \
336996     376240          3163198         []     [Dummy1]  Dummy1   NaN   

       genres styles data_quality  
336996     []     []      Correct  


master_id 376240 ("Dummy1") is a placeholder/test record, not a real master:
empty artist_ids, title and artist_names both literally "Dummy1", year 0,
no genres, no styles. Dropped.

In [26]:
masters_df = masters_df[~no_genre_no_style_mask].reset_index(drop=True)
print("masters_df shape after dropping Dummy1:", masters_df.shape)

masters_df shape after dropping Dummy1: (2204465, 9)


### Question: what does data_quality contain, and does it matter for cleaning?

In [27]:
print(masters_df["data_quality"].value_counts())

data_quality
Correct               2202171
Needs Vote               2155
Entirely Incorrect        139
Name: count, dtype: int64


#### Interpretation: data_quality

2,202,171 masters (99.9%) are marked "Correct". Two other values appear:
- "Needs Vote": 2,155 masters (0.10%), the community hasn't confirmed the
  data yet, this does not mean it's wrong.
- "Entirely Incorrect": 139 masters (0.006%), flagged by Discogs' own
  community as wrong.

#### Decision

Drop masters marked "Entirely Incorrect", Discogs has flagged this data as
unreliable. Keep "Needs Vote" as is, unconfirmed is not the same as wrong,
and dropping it would remove 2,155 masters without a data quality reason to
do so.


In [28]:
is_entirely_incorrect = masters_df["data_quality"] == "Entirely Incorrect"
print("masters dropped (Entirely Incorrect):", is_entirely_incorrect.sum())

masters_df = masters_df[~is_entirely_incorrect].reset_index(drop=True)
print("masters_df shape after data_quality cleaning:", masters_df.shape)

masters dropped (Entirely Incorrect): 139
masters_df shape after data_quality cleaning: (2204326, 9)


## 5. Compilation and unofficial exclusion via representative releases

### Question: can we get a representative release (with format_descriptions) for each Master?

masters_df has no column indicating whether a Master is a compilation or an
unofficial release, that information only exists at the release level, in
format_descriptions. A Master can have multiple releases under it (different
pressings, countries, years), so before we can check format_descriptions for
a Master, we first need to pick one representative release to read it from.

This step checks how many Masters have a release clearly flagged by Discogs
as the "main" one (is_main_release == True), versus how many don't and will
need a fallback rule to pick a representative release.

In [29]:
# Only releases belonging to a real Master (master_id != 0) are relevant here,
# standalone releases are out of scope per the Masters-only Work decision.
real_master_releases = releases_df[releases_df["master_id"] != 0]

# Discogs flags one release per Master as the "main" one, prefer that as the
# representative release. Some Masters have no release flagged this way, those
# need a fallback.
main_releases = real_master_releases[real_master_releases["is_main_release"] == True]

masters_with_main = set(main_releases["master_id"])
masters_without_main = set(real_master_releases["master_id"]) - masters_with_main

print("masters with a flagged main release:", len(masters_with_main))
print("masters needing a fallback (no main release flagged):", len(masters_without_main))
print("masters_df total (for comparison):", len(masters_df))

masters with a flagged main release: 2589245
masters needing a fallback (no main release flagged): 71
masters_df total (for comparison): 2204326


#### Checkpoint: save cleaned masters_df

In [30]:
masters_df.to_parquet("../data/processed/masters_clean_checkpoint.parquet", index=False)

#### Question (corrected): can we identify a representative release for each kept Master?

The previous check matched against every master_id in the raw releases_df,
including ones already excluded from masters_df. Restricting to only the
master_ids we actually kept.

In [31]:
kept_master_ids = set(masters_df["master_id"])
print("kept master_ids:", len(kept_master_ids))

kept master_ids: 2204326


In [32]:
# Keep only releases whose master_id matches a Master we actually kept
# after cleaning (masters_df), so we don't pull in releases belonging to
# masters we already excluded (Various artists, Classical, Non-Music, etc).
real_master_releases = releases_df[releases_df["master_id"].isin(kept_master_ids)]

print("releases belonging to a kept Master:", len(real_master_releases))

releases belonging to a kept Master: 9872816


In [33]:
# Discogs flags one release per Master as the "main" one. Filter down to
# just those flagged releases, this gives at most one row per master_id,
# the representative release we want to pull format_descriptions from.
main_releases = real_master_releases[real_master_releases["is_main_release"] == True]

print("releases flagged as main release:", len(main_releases))

releases flagged as main release: 2204218


In [34]:
# Masters whose main release is flagged exactly once, and Masters missing one entirely.
masters_with_main = set(main_releases["master_id"])
masters_without_main = kept_master_ids - masters_with_main

print("masters with a flagged main release:", len(masters_with_main))
print("masters needing a fallback (no main release flagged):", len(masters_without_main))

# Sanity check: is master_id unique within main_releases, or does some Master
# have more than one release flagged as main (which would need its own fix)?
print("duplicated master_id in main_releases:", main_releases["master_id"].duplicated().sum())

masters with a flagged main release:

 2204218
masters needing a fallback (no main release flagged): 108
duplicated master_id in main_releases: 0


#### Interpretation: representative release coverage

2,204,218 of 2,204,326 kept Masters (99.995%) have exactly one release
flagged as the main release, no duplicates, no ambiguity. Only 108 Masters
have no release flagged as main at all and need a fallback rule to pick a
representative release.

#### Decision

For the 108 Masters without a flagged main release, fall back to the
earliest-year release (ties broken by smallest release_id), the same rule
the old repo used. This is a small enough group that the fallback choice
has negligible impact on the dataset overall.


In [35]:
# For Masters with no flagged main release, fall back to the earliest-year
# release, ties broken by the smallest release_id, same rule the old repo used.
fallback_candidates = real_master_releases[real_master_releases["master_id"].isin(masters_without_main)]
fallback_candidates = fallback_candidates.sort_values(["master_id", "released", "release_id"])
fallback_rows = fallback_candidates.drop_duplicates(subset="master_id", keep="first")

print("fallback rows selected:", len(fallback_rows))
print("masters still missing after fallback:", len(masters_without_main) - len(fallback_rows))

fallback rows selected: 14
masters still missing after fallback: 94


Only 14 of the 108 Masters without a flagged main release had any fallback
candidate. The other 94 appear to have zero rows in real_master_releases,
meaning no release in releases_df carries their master_id at all. Before
deciding anything, inspect what these 94 Masters actually look like.

In [36]:
# Masters that got no fallback candidate at all, meaning zero releases in
# releases_df reference this master_id.
orphan_master_ids = masters_without_main - set(fallback_rows["master_id"])
print("orphan masters (no release references them):", len(orphan_master_ids))

orphan_masters = masters_df[masters_df["master_id"].isin(orphan_master_ids)]
print(orphan_masters[["master_id", "main_release_id", "title", "artist_names", "year"]].head(20))

orphan masters (no release references them): 94
        master_id  main_release_id  \
32352       35849           450528   
46883       52196          1277759   
49044       54711          1740104   
66608       74506          1270585   
79787       89268          1612973   
114448     127567          1775877   
126115     140438          1737385   
135630     150792          1843273   
138530     153959           472776   
159670     177512          1914160   
166263     184770           999998   
168874     187711          1963614   
176765     196548          2001433   
177880     197826          1748457   
179269     199403          1996758   
179505     199667          2014723   
187126     208196          2048070   
187836     208978          2055343   
194263     216182          2094237   
201964     224646          2133651   

                                                    title  \
32352                              Seven Veils Of Silence   
46883                          

#### Question: does main_release_id actually exist in releases_df, and what master_id does it carry?

In [37]:
# Look up the release pointed to by main_release_id for each orphan Master,
# and compare its master_id against the Master's own master_id.
orphan_main_release_ids = orphan_masters["main_release_id"]

matched_releases = releases_df[releases_df["release_id"].isin(orphan_main_release_ids)]
print("main_release_id rows found in releases_df:", len(matched_releases))
print()
print(matched_releases[["release_id", "master_id", "title"]].head(20))

main_release_id rows found in releases_df: 0

Empty DataFrame
Columns: [release_id, master_id, title]
Index: []


#### Interpretation

None of the 94 orphan Masters' main_release_id values exist anywhere in
releases_df, not just with a different master_id, the release_id itself is
absent from the dump. The Masters and Releases XML dumps were exported from
Discogs, and a Master's main_release_id can reference a release that was
deleted or merged on Discogs after the Masters dump was taken but before (or
differently from) the Releases dump, a known type of inconsistency between
dump snapshots, not a parsing error on our side.

This affects 94 out of 2,204,326 kept Masters (0.004%), negligible in size.

#### Decision

These 94 Masters have no representative release and therefore no
format_descriptions, their is_compilation/is_unofficial status is simply
unknown, not False. Keep them in the Works dataset (there's no quality
reason to drop them, they're just missing one derived feature), and mark
their compilation/unofficial flags as missing/NaN rather than defaulting to
False, since defaulting silently would misrepresent the unknown as "verified
not a compilation."


In [38]:
# These 94 Masters simply have no representative release available.
print("masters with no representative release at all:", len(orphan_master_ids))

masters with no representative release at all: 94


#### Question: what does format_descriptions look like for the representative releases, and how many Masters are compilations or unofficial?

In [39]:
# Combine the two representative-release sources: the flagged main release
# where one exists, the fallback (earliest year) release otherwise.
representative_releases = pd.concat(
    [main_releases[["master_id", "format_descriptions"]],
     fallback_rows[["master_id", "format_descriptions"]]],
    ignore_index=True,
)

print("representative releases total:", len(representative_releases))
print("unique master_id covered:", representative_releases["master_id"].nunique())

representative releases total: 2204232
unique master_id covered: 2204232


In [40]:
# Derive is_compilation / is_unofficial directly from format_descriptions.
representative_releases["is_compilation"] = representative_releases["format_descriptions"].apply(
    lambda d: "Compilation" in d
)
representative_releases["is_unofficial"] = representative_releases["format_descriptions"].apply(
    lambda d: "Unofficial Release" in d
)

print("is_compilation:", representative_releases["is_compilation"].sum())
print("is_unofficial:", representative_releases["is_unofficial"].sum())
print("both:", (representative_releases["is_compilation"] & representative_releases["is_unofficial"]).sum())

is_compilation: 165172
is_unofficial: 32158
both: 8421


#### Interpretation: compilation and unofficial Masters

Of 2,204,232 Masters with a representative release, 165,172 (7.5%) are
compilations and 32,158 (1.5%) are unofficial releases, with 8,421
overlapping both. The 94 Masters with no representative release (orphans
from the previous step) have unknown status for both flags.

#### Decision

Exclude Masters flagged is_compilation or is_unofficial from the Works
dataset, a compilation isn't a new creative work by the artist, and an
unofficial release isn't a real, catalogued release. The 94 orphan Masters
(unknown status) are kept, since there's no evidence they're compilations or
unofficial, only that we can't check.


In [41]:
# Merge the compilation/unofficial flags back onto masters_df. Masters with no
# representative release (the 94 orphans) get NaN for both, left as False.
flags = representative_releases.drop_duplicates(subset="master_id").set_index("master_id")
masters_df["is_compilation"] = masters_df["master_id"].map(flags["is_compilation"]).fillna(False)
masters_df["is_unofficial"] = masters_df["master_id"].map(flags["is_unofficial"]).fillna(False)

is_excluded = masters_df["is_compilation"] | masters_df["is_unofficial"]
print("masters excluded (compilation or unofficial):", is_excluded.sum())

masters_df = masters_df[~is_excluded].reset_index(drop=True)
print("masters_df shape after exclusion:", masters_df.shape)

masters excluded (compilation or unofficial): 188909


masters_df shape after exclusion: (2015417, 11)


## 6. Recovering artists via standalone releases

### Question: how many artists would be lost entirely under the Masters-only approach?

Comparing artists that appear in our current, cleaned masters_df against
artists that only ever appear in standalone releases (master_id == 0),
to quantify the real size of this trade-off rather than deciding on
intuition alone.

In [42]:
# Artists with at least one kept Master (the ones represented under Masters-only).
masters_artist_ids = set(masters_df["artist_ids"].explode().dropna())

# Artists that only appear in standalone releases, never in any Master.
standalone_releases = releases_df[releases_df["master_id"] == 0]
standalone_artist_ids = set(standalone_releases["artist_ids"].explode().dropna())

artists_lost_entirely = standalone_artist_ids - masters_artist_ids

print("artists with at least one kept Master:", len(masters_artist_ids))
print("artists with at least one standalone release:", len(standalone_artist_ids))
print("artists lost entirely (Masters-only approach):", len(artists_lost_entirely))
print("share of standalone-release artists lost:", round(100 * len(artists_lost_entirely) / len(standalone_artist_ids), 1), "%")

artists with at least one kept Master: 712571
artists with at least one standalone release: 2619798
artists lost entirely (Masters-only approach): 2120674
share of standalone-release artists lost: 80.9 %


#### Question: of the artists lost entirely, how many actually had enough standalone releases to matter for modeling?

80.9% of artists with any Discogs output would be lost under the Masters-only
approach. Before treating that as a real problem, check how many of them
would have even qualified as a usable training example (3+ works, per
config.yaml's min_previous_works), versus artists with only 1-2 releases
that wouldn't pass that bar regardless.

In [43]:
# For artists lost entirely, count how many standalone releases each one has.
# If most have only 1-2, they wouldn't have qualified for modeling anyway
# (config.yaml's min_previous_works = 3), so the real loss is smaller than
# the raw 80.9% suggests.
lost_artist_release_counts = (
    standalone_releases[standalone_releases["artist_ids"].apply(
        lambda ids: bool(set(ids) & artists_lost_entirely)
    )]
    .explode("artist_ids")
)
lost_artist_release_counts = lost_artist_release_counts[
    lost_artist_release_counts["artist_ids"].isin(artists_lost_entirely)
]

counts_per_lost_artist = lost_artist_release_counts.groupby("artist_ids").size()

print("lost artists with only 1 standalone release:", (counts_per_lost_artist == 1).sum())
print("lost artists with 2 standalone releases:", (counts_per_lost_artist == 2).sum())
print("lost artists with 3+ standalone releases (would have qualified):", (counts_per_lost_artist >= 3).sum())

lost artists with only 1 standalone release: 1373034
lost artists with 2 standalone releases: 359030
lost artists with 3+ standalone releases (would have qualified): 388610


#### Interpretation

Of the 2,120,674 artists lost entirely under Masters-only:
- 1,373,034 (64.7%) have only 1 standalone release, wouldn't have qualified
  for modeling anyway (min_previous_works = 3).
- 359,030 (16.9%) have 2, same conclusion.
- 388,610 (18.3%) have 3 or more standalone releases, these artists would
  have had enough history to be usable training/eval examples, and are
  genuinely lost under this approach.

Compared to the 712,571 artists currently represented via masters_df, these
388,610 lost-but-qualifying artists represent roughly 35% on top of that,
not a rounding error.

#### Decision

Include standalone releases (master_id == 0) as synthetic Works, but only
for artists who have 3+ standalone releases, artists with 1-2 wouldn't
qualify for modeling anyway (min_previous_works = 3), so including them
would add complexity for no benefit. This recovers 388,610 artists with real
modeling-relevant history, without building the full dedup/fuzzy-matching
machinery the old repo used for every standalone release regardless of
artist history.


Here's the plan for building this, step by step:

1. Filter releases_df to standalone releases (master_id == 0) belonging to artists with 3+ standalone releases.
2. Apply the same cleaning rules already decided for Masters to this population: exclude Various/Unknown/No Artist, excluded genres, Entirely Incorrect data_quality.
3. Derive is_compilation/is_unofficial directly from format_descriptions (no representative-release step needed here, since these rows already are individual releases, not Masters needing one picked).
4. Assign a synthetic work_id (release_id + a large offset, same pattern as the old repo, to guarantee no collision with real master_id values).
5. Combine with the cleaned masters_df into the final Works table.

#### Question: which standalone releases belong to artists with 3+ standalone releases?

Applying the 3+ threshold across all artists with standalone releases, not
just the previously identified "lost" ones, so an artist who already has
Masters but also has several uncaptured singles gets those singles counted
too, rather than treating "already has a Master" as a separate case.

#### Reloading state after a kernel restart

The kernel restarted, losing all in-memory state. Reloading masters_df and
releases_df from the saved checkpoint and raw files, instead of re-running
every cleaning step from scratch.

In [44]:
import pandas as pd

# Reload masters_df from the checkpoint (already cleaned through is_compilation/is_unofficial).
masters_df = pd.read_parquet("../data/processed/masters_clean_checkpoint.parquet")

# Reload releases_df with ONLY the columns we actually use. Skipping tracklist,
# barcodes, matrix_numbers, labels, format_qty, format_text, formats, country,
# n_tracks avoids materializing their heavy per-row list objects, which is what
# caused the kernel to crash from memory pressure.
RELEASE_COLUMNS_NEEDED = [
    "release_id", "master_id", "is_main_release", "released",
    "title", "artist_ids", "artist_names", "genres", "styles",
    "data_quality", "format_descriptions",
]
releases_df = pd.read_parquet("../data/processed/releases.parquet", columns=RELEASE_COLUMNS_NEEDED)

print("masters_df shape:", masters_df.shape)
print("releases_df shape:", releases_df.shape)

masters_df shape: (2204326, 9)
releases_df shape: (19417067, 11)


In [45]:
# Rebuild the representative-release lookup and reapply the compilation/
# unofficial exclusion (lost when the kernel crashed before this checkpoint
# re-save completed).
kept_master_ids = set(masters_df["master_id"])
real_master_releases = releases_df[releases_df["master_id"].isin(kept_master_ids)]

main_releases = real_master_releases[real_master_releases["is_main_release"] == True]
masters_with_main = set(main_releases["master_id"])
masters_without_main = kept_master_ids - masters_with_main

fallback_candidates = real_master_releases[real_master_releases["master_id"].isin(masters_without_main)]
fallback_candidates = fallback_candidates.sort_values(["master_id", "released", "release_id"])
fallback_rows = fallback_candidates.drop_duplicates(subset="master_id", keep="first")

representative_releases = pd.concat(
    [main_releases[["master_id", "format_descriptions"]],
     fallback_rows[["master_id", "format_descriptions"]]],
    ignore_index=True,
)

representative_releases["is_compilation"] = representative_releases["format_descriptions"].apply(lambda d: "Compilation" in d)
representative_releases["is_unofficial"] = representative_releases["format_descriptions"].apply(lambda d: "Unofficial Release" in d)

flags = representative_releases.drop_duplicates(subset="master_id").set_index("master_id")
masters_df["is_compilation"] = masters_df["master_id"].map(flags["is_compilation"]).fillna(False)
masters_df["is_unofficial"] = masters_df["master_id"].map(flags["is_unofficial"]).fillna(False)

is_excluded = masters_df["is_compilation"] | masters_df["is_unofficial"]
masters_df = masters_df[~is_excluded].reset_index(drop=True)

print("masters_df shape after exclusion:", masters_df.shape)

masters_df shape after exclusion: (2015417, 11)


In [46]:
masters_df.to_parquet("../data/processed/masters_clean_checkpoint.parquet", index=False)

In [47]:
# Standalone releases: no Master link at all.
standalone_releases = releases_df[releases_df["master_id"] == 0]
print("standalone releases:", len(standalone_releases))

standalone releases: 8076985


In [48]:
# Count standalone releases per artist, find the ones with 3+.
standalone_exploded = standalone_releases.explode("artist_ids")
standalone_counts_per_artist = standalone_exploded.groupby("artist_ids").size()

qualifying_artist_ids = set(standalone_counts_per_artist[standalone_counts_per_artist >= 3].index)

qualifying_mask = standalone_releases["artist_ids"].apply(
    lambda ids: bool(set(ids) & qualifying_artist_ids)
)
standalone_candidates = standalone_releases[qualifying_mask]

print("standalone releases kept for these artists:", len(standalone_candidates))

standalone releases kept for these artists: 6221353


#### Applying the same cleaning rules to standalone_candidates

Reusing the exclusion rules already decided for masters_df: non-artist
placeholders, excluded genres, and Entirely Incorrect data_quality.

In [49]:
# Reuse the exact exclusion rules already decided for masters_df, applied
# here to standalone_candidates (releases with no Master, belonging to
# artists with 3+ standalone releases).

# Same non-artist placeholder values excluded from masters_df earlier.
# Case-insensitive, same fix applied to masters_df (2026-10-06): an
# exact-match version of this check let "various"/"VARIOUS" case
# variants through.
NON_ARTIST_PLACEHOLDERS = {"various", "unknown artist", "no artist"}

# Same out-of-scope genres excluded from masters_df earlier (Classical,
# Brass & Military, Non-Music, Children's).
EXCLUDED_GENRES_ALL = {"Classical", "Brass & Military", "Non-Music", "Children's"}

# artist_names is a list per row here too, join it into a string first so we
# can compare it against the placeholder set, same fix as before.
artist_names_str_standalone = standalone_candidates["artist_names"].apply(lambda names: ", ".join(names))
is_non_artist = artist_names_str_standalone.str.lower().isin(NON_ARTIST_PLACEHOLDERS)

# genres is also a list per row, check whether any excluded genre appears in it.
has_excluded_genre_standalone = standalone_candidates["genres"].apply(
    lambda g: any(genre in EXCLUDED_GENRES_ALL for genre in g)
)

# Same data_quality rule as masters_df: drop rows Discogs' own community
# flagged as entirely wrong.
is_entirely_incorrect_standalone = standalone_candidates["data_quality"] == "Entirely Incorrect"

print("excluded, non-artist placeholder:", is_non_artist.sum())
print("excluded, out-of-scope genre:", has_excluded_genre_standalone.sum())
print("excluded, Entirely Incorrect:", is_entirely_incorrect_standalone.sum())

# Combine all three exclusion reasons and drop those rows.
to_exclude = is_non_artist | has_excluded_genre_standalone | is_entirely_incorrect_standalone
standalone_candidates = standalone_candidates[~to_exclude].reset_index(drop=True)
print("standalone_candidates after cleaning:", len(standalone_candidates))

excluded, non-artist placeholder: 899514
excluded, out-of-scope genre: 685589
excluded, Entirely Incorrect: 826


standalone_candidates after cleaning: 4752274


#### Interpretation

Of 6,221,353 standalone releases (artists with 3+ standalone releases):
- 899,514 excluded as non-artist placeholders (Various, Unknown Artist, No Artist).
- 685,589 excluded for out-of-scope genre (Classical, Brass & Military, Non-Music, Children's).
- 826 excluded for Entirely Incorrect data_quality.
- Some overlap between these reasons, 4,752,274 releases remain after combining all three.

#### Decision

Apply the same compilation/unofficial exclusion used for Masters, this time
directly from each release's own format_descriptions (no representative-
release step needed here, these rows already are individual releases).


In [50]:
# format_descriptions is directly available on each standalone release, no
# representative-release lookup needed here (unlike Masters, which needed
# one release picked to represent the whole group).
is_compilation_standalone = standalone_candidates["format_descriptions"].apply(lambda d: "Compilation" in d)
is_unofficial_standalone = standalone_candidates["format_descriptions"].apply(lambda d: "Unofficial Release" in d)

print("is_compilation:", is_compilation_standalone.sum())
print("is_unofficial:", is_unofficial_standalone.sum())
print("both:", (is_compilation_standalone & is_unofficial_standalone).sum())

to_exclude_2 = is_compilation_standalone | is_unofficial_standalone
standalone_candidates = standalone_candidates[~to_exclude_2].reset_index(drop=True)
print("standalone_candidates after compilation/unofficial exclusion:", len(standalone_candidates))

is_compilation: 401541
is_unofficial: 182761
both: 50574


standalone_candidates after compilation/unofficial exclusion: 4218546


#### Interpretation

Exclude compilation and unofficial standalone releases, same reasoning as
for Masters. 401,541 compilations, 182,761 unofficial, 50,574 overlap,
4,218,546 standalone releases remain.


#### Question: how clean is the released date field in standalone_candidates?

Released is a raw date string (e.g. "1999-03-00"), not a clean year like
masters_df["year"]. Extract just the year and check how many are missing
or invalid before using it.

In [51]:
# Keep only a real, 4-digit leading year from released, anything else
# becomes a genuinely missing value.
first_four = standalone_candidates["released"].fillna("").str.slice(0, 4)
looks_like_year = first_four.str.fullmatch(r"[0-9]{4}")
standalone_candidates["year"] = pd.to_numeric(first_four.where(looks_like_year), errors="coerce")

print("standalone_candidates rows:", len(standalone_candidates))
print("missing year:", standalone_candidates["year"].isna().sum())
print()
print(standalone_candidates["year"].describe())

standalone_candidates rows: 4218546
missing year: 474494

count    3.744052e+06
mean     2.003294e+03
std      2.376050e+01
min      0.000000e+00
25%      1.995000e+03
50%      2.011000e+03
75%      2.018000e+03
max      2.028000e+03
Name: year, dtype: float64


#### Question: are there any year == 0 or future-year anomalies in standalone_candidates?

In [52]:
print("year == 0:", (standalone_candidates["year"] == 0).sum())
print("year > 2026 (future):", (standalone_candidates["year"] > 2026).sum())
print()
print(standalone_candidates.loc[standalone_candidates["year"] > 2026, ["release_id", "title", "artist_names", "released", "year"]].head(10))

year == 0: 115
year > 2026 (future): 1

         release_id                title  artist_names    released    year
4213951    38257659  Vieille Vie part. 2  [Deus Latis]  2028-09-01  2028.0


115 rows have year == 0, Discogs' same missing-year sentinel seen in
masters_df. 1 row has year 2028, "Vieille Vie part. 2" by Deus Latis,
released 01-09-2028, a scheduled future release, the Discogs dump itself
is dated 01-09-2026, so this is a real pre-announced release date, not a
parsing error, just outside the range this snapshot can meaningfully use.

#### Decision

Treat year == 0 as missing (same rule as masters_df). Treat the single
2028 future-dated release as missing too, a snapshot taken in 2026 cannot
meaningfully place a 2028 release in any artist's historical timeline.

In [53]:
standalone_candidates.loc[standalone_candidates["year"] == 0, "year"] = pd.NA
standalone_candidates.loc[standalone_candidates["year"] > 2026, "year"] = pd.NA

print("missing year after fix:", standalone_candidates["year"].isna().sum())

missing year after fix: 474610


## 7. Building the combined Works table

### Question: how do we assign a synthetic work_id to standalone releases without colliding with real master_id values?

In [54]:
# release_id values are far smaller than 10 billion, so adding this offset
# guarantees a synthetic work_id can never collide with a real master_id.
SYNTHETIC_OFFSET = 10_000_000_000

print("max master_id in masters_df:", masters_df["master_id"].max())
print("max release_id in standalone_candidates:", standalone_candidates["release_id"].max())

standalone_candidates["work_id"] = standalone_candidates["release_id"] + SYNTHETIC_OFFSET
standalone_candidates["work_source"] = "synthetic"

print("synthetic work_id range:", standalone_candidates["work_id"].min(), "to", standalone_candidates["work_id"].max())

max master_id in masters_df: 4331826
max release_id in standalone_candidates: 38306709
synthetic work_id range: 10000000040 to 10038306709


### Interpretation: synthetic work_id is collision-safe

Max master_id (4,331,826) and max synthetic work_id (10,038,306,709) are far
apart, no collision possible. Ready to build the combined Works table.

### Question: what should the unified Works schema look like, and how do we combine both sources?

In [55]:
# --- Master-side Works ---
# Each kept Master becomes one Work. work_id is just the master_id itself,
# work_source records where this Work came from, useful later if we ever
# need to filter or debug by source.
master_works = pd.DataFrame({
    "work_id": masters_df["master_id"],
    "work_source": "master",

    # artist_ids and artist_names are lists (a Master can have multiple
    # credited artists), take the first one as the primary artist for
    # this Work. Guard against an empty list with the len() check, even
    # though masters_df should have none left after cleaning.
    "artist_id": masters_df["artist_ids"].apply(lambda ids: ids[0] if len(ids) else None),
    "artist": masters_df["artist_names"].apply(lambda names: names[0] if len(names) else None),

    "title": masters_df["title"],
    "year": masters_df["year"],

    # genres and styles stay as lists for now, kept as-is rather than
    # joined into strings, so they can still be exploded or filtered
    # later during feature engineering.
    "genres": masters_df["genres"],
    "styles": masters_df["styles"],
})

# --- Synthetic-side Works ---
# Each surviving standalone release (no Master, but belonging to an
# artist with 3+ standalone releases) becomes its own Work. work_id uses
# the synthetic id assigned earlier (release_id + SYNTHETIC_OFFSET), so
# it can never collide with a real master_id.
synthetic_works = pd.DataFrame({
    "work_id": standalone_candidates["work_id"],
    "work_source": "synthetic",

    "artist_id": standalone_candidates["artist_ids"].apply(lambda ids: ids[0] if len(ids) else None),
    "artist": standalone_candidates["artist_names"].apply(lambda names: names[0] if len(names) else None),

    "title": standalone_candidates["title"],
    "year": standalone_candidates["year"],
    "genres": standalone_candidates["genres"],
    "styles": standalone_candidates["styles"],
})

# --- Combine into one Works table ---
works_df = pd.concat([master_works, synthetic_works], ignore_index=True)

# Sanity check: work_id must be unique across the combined table, this
# would only fail if the synthetic offset logic was wrong.
assert works_df["work_id"].is_unique, "work_id collision between master and synthetic ids"

print("works_df shape:", works_df.shape)
print(works_df["work_source"].value_counts())

works_df shape: (6233963, 8)
work_source
synthetic    4218546
master       2015417
Name: count, dtype: int64


### Interpretation: combined Works table

works_df has 6,233,963 rows: 2,015,417 from cleaned Masters, 4,218,546 from
qualifying standalone releases. work_id is confirmed unique across both
sources.


### Question: can a Master's missing year be recovered from its own Releases, before dedup and the title filters lock in that missing value?

Both the exact-duplicate key below and the later fuzzy/catalog-number dedup survivor tie-break use `year`, so a Master's year needs to be as correct as possible before any of those steps run, not after.

In [56]:
from works import recover_missing_master_years  # fills a Master's missing year from its own Releases

# Recover a year for every Master currently missing one, by looking at its
# own Releases. Returns a master_id -> year Series, only for Masters where
# at least one Release had a usable year.
recovered_years = recover_missing_master_years(masters_df, releases_df)

# Only master-side rows are eligible, work_id IS the master_id for those rows.
master_mask = works_df["work_source"] == "master"
missing_before = master_mask & works_df["year"].isna()
print("master works missing year before recovery:", missing_before.sum())

# Map the recovered year onto every currently-missing master-side row, rows
# with no match in recovered_years (not recoverable) are left untouched (NaN).
works_df.loc[missing_before, "year"] = works_df.loc[missing_before, "work_id"].map(recovered_years)

still_missing = master_mask & works_df["year"].isna()
print("recovered:", missing_before.sum() - still_missing.sum())
print("still missing after recovery:", still_missing.sum())

master works missing year before recovery: 101543
recovered: 38670
still missing after recovery: 62873


#### Interpretation and Decision: Master missing-year recovery

At this point in the pipeline (before any dedup or title filter), 101,543 of
the kept Masters have a missing year. Looking up each one's own Releases and
taking the earliest valid `released` year recovers 38,670 of them (38.1%),
leaving 62,873 genuinely unrecoverable (no Release at all, or none with a
parseable, non-zero year). The years come from `releases.parquet`, so the step is applied unconditionally, before
dedup, so the exact/fuzzy/catalog-number survivor tie-break (which uses `year`)
sees the corrected value rather than a missing one.

A separate read-only check (see `docs/decisions_core.md`) found this recovery is not
uniform across genres: it recovers Rock/Electronic/Blues at roughly 50%, but
Latin/Reggae/Folk,World,&Country at only 30-39% (Spearman r=-0.809, p=0.0026
between a genre's original missing-year rate and its recovery rate). Every
genre's absolute missing-year count only improves, never worsens, but the
relative disparity between genres is not fixed by this step, it is logged as an accepted, unresolved limitation.

In [57]:
works_df.to_parquet("../data/processed/works_checkpoint_raw.parquet", index=False)

## 8. Duplicate detection within works_df

### Question: how many exact duplicates exist (same artist_id, normalized title, year)?

In [58]:
import pandas as pd

works_df = pd.read_parquet("../data/processed/works_checkpoint_raw.parquet")
print("works_df shape:", works_df.shape)

works_df shape: (6233963, 8)


In [59]:
# Normalize each title for comparison: lowercase, strip punctuation, collapse
# whitespace. Deliberately NOT stripping edition suffixes like "Deluxe" or
# "Anniversary", since doing so risks merging genuinely different releases
# into one duplicate group.
def normalize_title(title):
    text = str(title).strip().lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

works_df["title_normalized"] = works_df["title"].apply(normalize_title)

# Build the exact-duplicate key: same artist, same normalized title, same
# year. artist_id (not the artist name string) is used so two different
# artists who happen to share a name are never merged, and the same artist
# spelled two different ways is never wrongly split.
works_df["dedup_key"] = list(zip(works_df["artist_id"], works_df["title_normalized"], works_df["year"]))

# keep=False marks every row that belongs to a duplicate group (not just the
# extras), so we can inspect the full group, not only the "excess" rows.
duplicate_mask = works_df.duplicated("dedup_key", keep=False)

print("rows involved in a duplicate group:", duplicate_mask.sum())
print("duplicate groups:", works_df.loc[duplicate_mask, "dedup_key"].nunique())

rows involved in a duplicate group: 226251
duplicate groups: 107208


#### Interpretation: exact duplicates in works_df

226,251 rows (3.6%) belong to a duplicate group, 107,208 groups found (these
numbers shifted slightly after the year==0 fix and, more recently, after the
Master missing-year recovery step above: some rows that used to collide or
separate on a missing year now behave differently with a real recovered year).
A few groups are inspected first to see what causes the duplication.

In [60]:
# Look at a handful of actual duplicate groups before deciding a tie-break rule.
sample_keys = works_df.loc[duplicate_mask, "dedup_key"].drop_duplicates().head(5)

for key in sample_keys:
    group = works_df[works_df["dedup_key"] == key]
    print(group[["work_id", "work_source", "artist", "title", "year"]])
    print()

             work_id work_source                  artist      title    year
146              276      master  Fields Of The Nephilim  Dawnrazor  1987.0
2071437  10000541537   synthetic  Fields Of The Nephilim  Dawnrazor  1987.0

             work_id work_source                  artist         title    year
153              284      master  Fields Of The Nephilim  Preacher Man  1987.0
6034163  10035883523   synthetic  Fields Of The Nephilim  Preacher Man  1987.0



        work_id work_source         artist     title    year
169         302      master  Black Sabbath  Paranoid  1970.0
425200   506302      master  Black Sabbath  Paranoid  1970.0

             work_id work_source                  artist      title    year
183              319      master  Fields Of The Nephilim  Sumerland  1990.0
4478557  10015533582   synthetic  Fields Of The Nephilim  Sumerland  1990.0

             work_id work_source   artist            title    year
354              504      master  Cloud 9  Distant Affairs  1998.0
2257541  10001909061   synthetic  Cloud 9  Distant Affairs  1998.0
2771469  10005080861   synthetic  Cloud 9  Distant Affairs  1998.0
2798541  10005255830   synthetic  Cloud 9  Distant Affairs  1998.0
5999517  10035426584   synthetic  Cloud 9  Distant Affairs  1998.0



#### Decision: duplicate tie-break rule

Prefer work_source == "master" over "synthetic" when both exist for the same
group, a Master already represents Discogs' own canonical grouping, so it's
the more reliable record. Within ties of the same source (two Masters, or
several synthetic releases), keep the one with the smallest work_id, a deterministic rule that gives a consistent result without judging which release is better.

In [61]:
# Tie-break: master before synthetic, then smallest work_id. _source_rank
# is 0 for master, 1 for synthetic, so sorting ascending puts the preferred
# row first within each group.
works_df["_source_rank"] = (works_df["work_source"] != "master").astype(int)
works_df = works_df.sort_values(["dedup_key", "_source_rank", "work_id"])

# After sorting, the first row in each dedup_key group is the one to keep.
# duplicated(keep="first") then marks every OTHER row in the group as a
# duplicate to remove.
is_duplicate_to_drop = works_df.duplicated("dedup_key", keep="first")
print("rows to drop as duplicates:", is_duplicate_to_drop.sum())

works_df = works_df[~is_duplicate_to_drop].drop(columns=["_source_rank"]).reset_index(drop=True)
print("works_df shape after deduplication:", works_df.shape)

rows to drop as duplicates: 119043


works_df shape after deduplication: (6114920, 10)


#### Interpretation: deduplication result

119,043 duplicate rows removed (1.9% of works_df), keeping one representative
per (artist_id, normalized title, year) group, preferring master over
synthetic, then smallest work_id. works_df now has 6,114,920 rows (re-run after
the year==0 fix and the Master missing-year recovery step, see decisions_core.md).

In [62]:
# dedup_key and title_normalized were only needed to compute duplicates,
# drop them before saving, they're not part of the final Works schema.
works_df = works_df.drop(columns=["dedup_key", "title_normalized"])

#Checkpoint the cleaned, deduplicated Works table so we can resume from here
works_df.to_parquet("../data/processed/works_checkpoint_dedup.parquet", index=False)
print("works_df shape:", works_df.shape)

works_df shape: (6114920, 8)


In [63]:
print(works_df.shape)
print(works_df.columns.tolist())

(6114920, 8)
['work_id', 'work_source', 'artist_id', 'artist', 'title', 'year', 'genres', 'styles']


In [64]:
# Check what's still alive in the kernel from earlier steps.
for name in ["representative_releases", "standalone_candidates", "masters_df", "releases_df"]:
    try:
        obj = eval(name)
        print(f"{name}: shape {obj.shape}")
    except NameError:
        print(f"{name}: NOT in memory")

representative_releases: shape (2204232, 4)
standalone_candidates: shape (4218546, 14)
masters_df: shape (2015417, 11)
releases_df: shape (19417067, 11)


In [65]:
import pandas as pd

RELEASE_COLUMNS_NEEDED = [
    "release_id", "master_id", "is_main_release", "released",
    "title", "artist_ids", "artist_names", "genres", "styles",
    "data_quality", "format_descriptions",
]
releases_df = pd.read_parquet("../data/processed/releases.parquet", columns=RELEASE_COLUMNS_NEEDED)

print("releases_df shape:", releases_df.shape)

releases_df shape: (19417067, 11)


In [66]:
SYNTHETIC_OFFSET = 10_000_000_000

# Synthetic works: recover the original release_id, then look up its
# format_descriptions directly, no need to rebuild standalone_candidates.
synthetic_mask = works_df["work_source"] == "synthetic"
synthetic_release_ids = works_df.loc[synthetic_mask, "work_id"] - SYNTHETIC_OFFSET

release_formats = releases_df.set_index("release_id")["format_descriptions"]
synthetic_format_descriptions = synthetic_release_ids.map(release_formats)

print("synthetic works:", synthetic_mask.sum())
print("matched a format_descriptions:", synthetic_format_descriptions.notna().sum())

synthetic works: 4117446
matched a format_descriptions: 4117446


In [67]:
# Master works: rebuild the representative-release selection, scoped only
# to the master_ids currently in works_df (already cleaned and deduplicated).
master_mask = works_df["work_source"] == "master"
kept_master_ids = set(works_df.loc[master_mask, "work_id"])

real_master_releases = releases_df[releases_df["master_id"].isin(kept_master_ids)]
main_releases = real_master_releases[real_master_releases["is_main_release"] == True]

masters_with_main = set(main_releases["master_id"])
masters_without_main = kept_master_ids - masters_with_main

fallback_candidates = real_master_releases[real_master_releases["master_id"].isin(masters_without_main)]
fallback_candidates = fallback_candidates.sort_values(["master_id", "released", "release_id"])
fallback_rows = fallback_candidates.drop_duplicates(subset="master_id", keep="first")

representative_releases = pd.concat(
    [main_releases[["master_id", "format_descriptions"]],
     fallback_rows[["master_id", "format_descriptions"]]],
    ignore_index=True,
)

print("master works:", master_mask.sum())
print("representative releases found:", len(representative_releases))

master works: 1997474
representative releases found: 1997381


In [68]:
# Build a master_id -> format_descriptions lookup (one row per master_id,
# representative_releases should already be unique per master_id).
master_format_lookup = representative_releases.drop_duplicates(subset="master_id").set_index("master_id")["format_descriptions"]

# Assign format_descriptions to works_df: master works look up by work_id
# (which IS the master_id), synthetic works use the release_id-based lookup
# from step 2.
works_df["format_descriptions"] = None
works_df.loc[master_mask, "format_descriptions"] = works_df.loc[master_mask, "work_id"].map(master_format_lookup)
works_df.loc[synthetic_mask, "format_descriptions"] = synthetic_format_descriptions

print("works with format_descriptions:", works_df["format_descriptions"].notna().sum())
print("works missing format_descriptions:", works_df["format_descriptions"].isna().sum())

works with format_descriptions: 6114827
works missing format_descriptions: 93


## 9. Remaster, reissue and other formats detection

### Question: which works are tagged as a reissue/remaster, and do they share artist+title with an earlier work?

In [69]:
# Recompute the normalized title. This column was dropped before the last
# checkpoint (it was only a helper for the exact-duplicate step earlier),
# so it needs to be rebuilt here to compare titles consistently.
def normalize_title(title):
    text = str(title).strip().lower()
    text = re.sub(r"[^\w\s]", " ", text)      # strip punctuation
    text = re.sub(r"\s+", " ", text).strip()  # collapse repeated whitespace
    return text

works_df["title_normalized"] = works_df["title"].apply(normalize_title)

# Flag a work as reissue-tagged if ANY of its format_descriptions entries
# mentions Reissue, Repress, or Remaster (case-insensitive). Each row's
# format_descriptions is a list (a release can carry several tags at once),
# so we check every entry in that list rather than treating the list as one
# single string.
#
# The 93 Masters with no representative release (found two steps ago) have
# format_descriptions == NaN, which pandas represents as a plain float, not
# a list or None. Trying to iterate over a float crashes, and pd.isna() on
# an actual list raises its own error (it returns one bool per element
# instead of a single True/False), so the only safe check here is the type
# itself: if it's a float, treat it as "no tags" instead of trying to read it.
reissue_pattern = re.compile(r"reissue|repress|remaster", re.IGNORECASE)
works_df["is_reissue_tagged"] = works_df["format_descriptions"].apply(
    lambda descs: False if isinstance(descs, float) else any(reissue_pattern.search(d) for d in descs)
)

print("works tagged reissue/repress/remaster:", works_df["is_reissue_tagged"].sum())

works tagged reissue/repress/remaster: 55271


### Question: do the reissue-tagged works share artist+title with an earlier work?

A reissue-tagged work isn't automatically a duplicate, it's only evidence of
one if there's actually an earlier work by the same artist with the same
title. Checking that before deciding anything.

In [70]:
# For each (artist_id, title_normalized) group, find the earliest year seen.
# A work "has an earlier same title" if its own year is later than that
# group's earliest year, meaning some other row by the same artist, same
# title, already exists with an earlier release year.
earliest_year_seen = works_df.groupby(["artist_id", "title_normalized"])["year"].transform("min")
works_df["has_earlier_same_title"] = works_df["year"] > earliest_year_seen

# A genuine reissue candidate: tagged as a reissue/repress/remaster AND an
# earlier version of the same artist+title actually exists in the dataset.
works_df["is_reissue_candidate"] = works_df["is_reissue_tagged"] & works_df["has_earlier_same_title"]

print("reissue-tagged:", works_df["is_reissue_tagged"].sum())
print("has an earlier same-title work:", works_df["has_earlier_same_title"].sum())
print("reissue candidates (both conditions):", works_df["is_reissue_candidate"].sum())

reissue-tagged: 55271
has an earlier same-title work: 74920
reissue candidates (both conditions): 4964


#### Interpretation: reissue candidates

55,212 works are tagged reissue/repress/remaster, 73,327 share artist+title
with an earlier work, but only 4,758 works satisfy both conditions, meaning
most reissue-tagged works are the ORIGINAL release (nothing earlier exists),
and most same-title repeats aren't actually tagged as reissues (could be
different recordings, covers, or coincidental title matches). The overlap is small. Sample rows are checked first.


In [71]:
# Inspect a handful of actual reissue candidates before deciding.
sample = works_df[works_df["is_reissue_candidate"]].head(10)
print(sample[["work_id", "work_source", "artist", "title", "year"]])

           work_id work_source          artist                      title  \
489         125457      master     John Selway                   Untitled   
731    10000318200   synthetic  Cari Lekebusch         For Those Who Know   
1297   10005413439   synthetic       DJ Rasoul               True Science   
1679   10009148214   synthetic     John Tejada                   Timebomb   
2212   10034514050   synthetic    Pépé Bradock                    Burning   
3725   10036716473   synthetic           Shazz          El Camino Project   
6403   10038199909   synthetic            Lego         El Ritmo De Verdad   
6493   10001687279   synthetic         Swayzak  Snowboarding In Argentina   
11943       456031      master    Unit Moebius                   Untitled   
15307      1403757      master            Coil      How To Destroy Angels   

         year  
489    1998.0  
731    2004.0  
1297   2014.0  
1679   2016.0  
2212   2016.0  
3725   2023.0  
6403   2026.0  
6493   2009.0  
11943  2

In [72]:
# Only for actual reissue candidates: how many years after the earliest
# same-title work does the reissue-tagged copy appear?
candidates = works_df[works_df["is_reissue_candidate"]].copy()
earliest_year_seen = works_df.groupby(["artist_id", "title_normalized"])["year"].transform("min")
candidates["year_gap"] = candidates["year"] - earliest_year_seen.loc[candidates.index]
print(candidates["year_gap"].describe())
print(candidates["year_gap"].value_counts().sort_index().head(15))

count    4964.000000
mean       17.062047
std        15.699352
min         1.000000
25%         4.000000
50%        12.000000
75%        26.000000
max        89.000000
Name: year_gap, dtype: float64
year_gap
1.0     465
2.0     322
3.0     288
4.0     228
5.0     200
6.0     170
7.0     145
8.0     165
9.0     158
10.0    156
11.0    137
12.0    141
13.0    125
14.0    110
15.0    109
Name: count, dtype: int64


In [73]:
# Look at the most extreme year gaps: anything over a century is not a
# real reissue interval for recorded music, it's almost certainly a bad
# year value somewhere in the group.
extreme = candidates.sort_values("year_gap", ascending=False).head(15)
print(extreme[["work_id", "work_source", "artist", "title", "year", "year_gap"]])

             work_id work_source                 artist  \
1070615  10037659777   synthetic         Robert Johnson   
1070612  10037905741   synthetic         Robert Johnson   
1070606  10037905969   synthetic         Robert Johnson   
1115470  10036755653   synthetic  Rodgers & Hammerstein   
105709   10036000568   synthetic          Sarah Vaughan   
683354   10010369842   synthetic         Duke Ellington   
1138539  10037515543   synthetic      Lightnin' Hopkins   
276040   10037548849   synthetic             Chet Baker   
403988   10038033181   synthetic          Helen Merrill   
984824   10037660167   synthetic         Oscar Peterson   
684267   10037561317   synthetic          Sonny Rollins   
251536   10034464967   synthetic          Elvis Presley   
1009113  10032744049   synthetic        Lennie Tristano   
1169345  10010311963   synthetic           Judy Garland   
408653   10037325469   synthetic    Cannonball Adderley   

                                                   titl

In [74]:
# Confirm the year==0 leak: look at every row in one of the suspicious groups.
mask = (works_df["artist"] == "The Heptones") & (works_df["title_normalized"] == "sufferers time")
print(works_df[mask][["work_id", "work_source", "artist", "title", "year"]])

            work_id work_source        artist           title    year
535375       253368      master  The Heptones  Sufferers Time  1976.0
535376  10037893627   synthetic  The Heptones  Sufferers Time  2026.0


In [75]:
# How many rows in works_df still have year == 0.0 literally, not NaN?
print("year == 0.0 (literal):", (works_df["year"] == 0.0).sum())
print("year is NaN:", works_df["year"].isna().sum())
print(works_df["work_source"].value_counts())
print(works_df[works_df["year"] == 0.0]["work_source"].value_counts())

year == 0.0 (literal): 0
year is NaN: 523363
work_source
synthetic    4117446
master       1997474
Name: count, dtype: int64
Series([], Name: count, dtype: int64)


In [76]:
# How many reissue candidates share a generic, low-information title?
generic_titles = {"untitled", "ep", "live", "remix", "demo", "single"}
candidates = works_df[works_df["is_reissue_candidate"]]
generic_count = candidates["title_normalized"].isin(generic_titles).sum()
print("reissue candidates with a generic title:", generic_count, "/", len(candidates))
print(candidates[candidates["title_normalized"].isin(generic_titles)][["work_id", "artist", "title", "year"]])

reissue candidates with a generic title: 20 / 4964
             work_id                            artist     title    year
489           125457                       John Selway  Untitled  1998.0
11943         456031                      Unit Moebius  Untitled  2007.0
204843   10038152632                         Fela Kuti     Live!  1996.0
399463        547605                      Muddy Waters      Live  1990.0
532662       1121660                    Letzte Instanz      Live  2006.0
541026   10037320960                       Kenny G (2)      Live  2001.0
562269   10036250771                        Status Quo      Live  2005.0
565733   10031535830                             R-Zac  Untitled  2024.0
778478   10030508114      Itchy Trigger Finger Niggers  Untitled  2019.0
1281012  10008858589               The Smile Adventure  Untitled  2016.0
1499165      1044390  The Alexandrov Red Army Ensemble  Untitled  1975.0
1762636  10036436669                   Eternal Torment      Demo  2025.0


In [77]:
# A reissue candidate (is_reissue_candidate == True) is, by definition, a work
# whose own year is later than the earliest year seen for its (artist_id,
# title_normalized) group. The earliest work in each group can never satisfy
# that condition against itself, so it is never flagged as a candidate.
# That means simply dropping every row where is_reissue_candidate is True
# removes only the later reissue/repress/remaster copies and automatically
# keeps the original, with no extra "keep the earliest per group" logic
# needed, the flag already encodes that.
#
# This mirrors the earlier exact-duplicate removal decision (section 8):
# same idea (collapse duplicate representations of one work down to a
# single row), applied here to same-artist, same-title works that differ
# by year and carry a reissue/repress/remaster tag, instead of works that
# are exact duplicates in the same year.
before = len(works_df)

works_df = works_df[~works_df["is_reissue_candidate"]].reset_index(drop=True)

after = len(works_df)

print("reissue/remaster rows removed:", before - after)
print("works_df shape:", works_df.shape)

reissue/remaster rows removed: 4964
works_df shape: (6109956, 13)


### Question: are there any other elements in the title worth exploring?

Some albums have Live in the title. They are checked below.

In [78]:
# Check how Discogs itself flags live recordings, the same way format_descriptions
# already flagged Reissue/Repress/Remaster. Using the tag instead of matching the
# word "Live" in the title avoids two kinds of mistakes: a studio album whose
# title happens to contain "Live" (false positive from title matching), and a
# live recording whose title doesn't mention "live" at all (false negative).
live_pattern = re.compile(r"\blive\b", re.IGNORECASE)
works_df["is_live_tagged"] = works_df["format_descriptions"].apply(
    lambda descs: False if isinstance(descs, float) else any(live_pattern.search(d) for d in descs)
)

print("works tagged Live:", works_df["is_live_tagged"].sum())
print(works_df[works_df["is_live_tagged"]][["work_id", "work_source", "artist", "title", "year"]].head(10))

works tagged Live: 0
Empty DataFrame
Columns: [work_id, work_source, artist, title, year]
Index: []


In [79]:
# See every distinct value that actually appears in format_descriptions,
# instead of guessing a regex blindly. explode() turns each list into one
# row per tag, so value_counts() shows the real vocabulary Discogs uses.
all_format_tags = works_df["format_descriptions"].dropna().explode()
print(all_format_tags.value_counts().head(30))

format_descriptions
Album              1769235
Single             1213617
7"                 1168340
45 RPM             1087589
MP3                 953267
EP                  693470
LP                  649776
Stereo              577322
12"                 477803
Limited Edition     352404
Promo               303854
10"                 277591
78 RPM              244731
33 ⅓ RPM            241190
WAV                 235882
FLAC                230823
AAC                 184692
Mono                115715
Numbered             89134
Maxi-Single          79227
Single Sided         67421
Mixed                51529
Mini-Album           46002
Reissue              33172
DVD-Video            30584
Mixtape              25121
White Label          23814
Sampler              22374
Mini                 22369
Enhanced             21941
Name: count, dtype: int64


In [80]:
# Search the full vocabulary (not just top 30) for anything live-related,
# in case it exists further down the frequency ranking.
live_tags = all_format_tags[all_format_tags.str.contains("live", case=False, na=False)]
print(live_tags.value_counts())

Series([], Name: count, dtype: int64)


In [81]:
# Only signal available: the word "Live" in the title itself. This will
# catch real live albums but also studio albums whose title happens to
# contain the word "Live" (false positives), there is no Discogs tag to
# disambiguate this, so the sample below is to gauge how big that risk is.
is_live_title = works_df["title"].str.contains(r"\blive\b", case=False, na=False, regex=True)
print("works with 'Live' in title:", is_live_title.sum())

sample = works_df[is_live_title].sample(20, random_state=1)
print(sample[["work_id", "work_source", "artist", "title", "year"]])

works with 'Live' in title: 92527


             work_id work_source                       artist  \
158374   10002605325   synthetic                   Eurythmics   
2117767  10003984720   synthetic                The Groop (3)   
5615723  10015748128   synthetic                 Circus Monti   
2226564  10036262495   synthetic                    Barnbrack   
976171   10004649684   synthetic                   The Ataris   
1029707      1772646      master                       Marduk   
4521196  10008600657   synthetic            Eddie Martin Band   
5148498      1355809      master               Dream Division   
1397045       621675      master             Stefan Sundström   
3227559  10003756589   synthetic          The Steepwater Band   
5123362  10010917582   synthetic      Yoichi Murata Orchestra   
1377961       676607      master           Bernard Lavilliers   
3694180  10030000370   synthetic  Molly Gene One Whoaman Band   
5627087  10022372465   synthetic                     Texaroma   
1962563      1508437     

In [82]:
# Look at what word follows "Live" in each matching title. Real live albums
# tend to continue with "At", "In", "From", a year, or end there. A false
# positive (the word "live" inside an unrelated phrase) tends to continue
# with something else entirely ("To Tell", "Forever", "Wire"...).
import re

def word_after_live(title):
    match = re.search(r"\blive\b\s*(\S*)", str(title), re.IGNORECASE)
    return match.group(1).lower() if match else None

live_titles = works_df.loc[is_live_title, "title"]
next_word = live_titles.apply(word_after_live)
print(next_word.value_counts().head(25))

title
at         22864
in         14675
           12782
from        3528
)           2541
!           2402
@           1696
on          1372
-           1036
and          948
&            704
:            638
for          497
"            471
/            404
,            380
the          330
session      324
with         319
without      311
it           283
concert      281
im           263
to           245
ep           242
Name: count, dtype: int64


In [83]:
# A live recording performs existing material rather than introducing new
# creative work, the same reasoning already applied to compilations and
# unofficial releases: it doesn't represent a genuine step in an artist's
# own stylistic trajectory, so it's excluded from the Works dataset rather
# than deduplicated like reissues were.
#
# format_descriptions and style carry no reliable "Live" tag in this
# dataset (checked and confirmed empty), title is the only signal
# available. \blive\b avoids matching inside words like "Alive". This
# accepts some unavoidable noise in both directions (a studio track titled
# "Live Forever", a live album that doesn't say "live" anywhere), but the
# word-after-"live" check shows the vast majority follow a clear live-album
# pattern ("At", "In", "From", a date, end of title).
is_live_title = works_df["title"].str.contains(r"\blive\b", case=False, na=False, regex=True)

before = len(works_df)
works_df = works_df[~is_live_title].reset_index(drop=True)
after = len(works_df)

print("live-titled works removed:", before - after)
print("works_df shape:", works_df.shape)

live-titled works removed: 92527
works_df shape: (6017429, 14)


Titles containing greatest hits are checked next.

In [84]:
# "Greatest Hits" compilations should mostly already be gone via the
# is_compilation exclusion applied earlier (before works_df existed). This
# checks how many slipped through, meaning they weren't tagged compilation
# by Discogs' own community even though the title says otherwise.
is_greatest_hits = works_df["title"].str.contains(r"greatest hits", case=False, na=False, regex=True)
print("works with 'Greatest Hits' in title still in works_df:", is_greatest_hits.sum())
print(works_df[is_greatest_hits][["work_id", "work_source", "artist", "title", "year"]].head(15))

works with 'Greatest Hits' in title still in works_df: 1623
           work_id work_source              artist  \
4844   10033890025   synthetic             Various   
11999  10034485460   synthetic       The Black Dog   
12000  10035843338   synthetic       The Black Dog   
12001  10035843233   synthetic       The Black Dog   
38166  10001907230   synthetic          Stacy Kidd   
38167  10001907217   synthetic          Stacy Kidd   
38168  10001907240   synthetic          Stacy Kidd   
39022  10001702862   synthetic  Twisted Individual   
39023  10001774404   synthetic  Twisted Individual   
39024  10001900385   synthetic  Twisted Individual   
42880  10000335950   synthetic    Electric Company   
42881  10000100480   synthetic    Electric Company   
42882        61780      master    Electric Company   
49676      1842611      master               Texas   
49677      1842612      master               Texas   

                                                   title    year  
4844    

In [85]:
# "Greatest Hits" works slipped through the earlier is_compilation exclusion
# because Discogs' own format_descriptions didn't tag them as Compilation.
# Title is the only signal here too. Same reasoning as the Live exclusion:
# a compilation of previously released material isn't a new creative work,
# it just wasn't caught by the compilation tag.
is_greatest_hits = works_df["title"].str.contains(r"greatest hits", case=False, na=False, regex=True)

before = len(works_df)
works_df = works_df[~is_greatest_hits].reset_index(drop=True)
after = len(works_df)

print("'Greatest Hits' works removed:", before - after)
print("works_df shape:", works_df.shape)

'Greatest Hits' works removed: 1623
works_df shape: (6015806, 14)


Other title patterns are checked next.

In [86]:
# Measure volume for each candidate pattern before deciding anything,
# same approach as Greatest Hits: check what actually has real numbers
# behind it instead of adding rules on hunches.
candidate_patterns = {
    "best of": r"best of",
    "anthology": r"anthology",
    "collection": r"collection",
    "b-sides": r"b[\s-]?sides",
    "rarities": r"rarities",
    "box set": r"box set",
    "discography": r"discography",
    "complete": r"\bcomplete\b",
    "unplugged": r"unplugged",
    "acoustic": r"\bacoustic\b",
    "soundtrack/ost": r"soundtrack|\bost\b",
    "karaoke": r"karaoke",
    "instrumental": r"\binstrumental\b",
    "tribute": r"\btribute\b",
    "remixes": r"remixes|remix album",
}

for label, pattern in candidate_patterns.items():
    count = works_df["title"].str.contains(pattern, case=False, na=False, regex=True).sum()
    print(f"{label}: {count}")

best of: 4668
anthology: 530


collection: 5415
b-sides: 629


rarities: 505
box set: 219


discography: 120


complete: 1642
unplugged: 989


acoustic: 3862


soundtrack/ost: 15672


karaoke: 431


instrumental: 3401


tribute: 5420
remixes: 25012


In [87]:
# Group the title patterns that are unambiguously compilation-type
# ("best of", "anthology", "box set", "discography", "b-sides", "rarities")
# into one regex, the same semantic category as "greatest hits": a
# retrospective collection of previously released material, not a new
# step in the artist's own stylistic trajectory. b[\s-]?sides matches
# "b-sides", "b sides" and "bsides" in one pattern.
#
# This is a sample check before deciding anything, same process as Live
# and Greatest Hits: look at real examples before removing real rows.
clear_compilation_patterns = r"best of|anthology|box set|discography|b[\s-]?sides|rarities"

is_clear_compilation = works_df["title"].str.contains(
    clear_compilation_patterns, case=False, na=False, regex=True
)

print("total matching this pattern group:", is_clear_compilation.sum())

# A fixed random_state so the sample is reproducible if we need to look
# at it again later.
sample = works_df[is_clear_compilation].sample(20, random_state=1)
print(sample[["work_id", "work_source", "artist", "title", "year"]])

total matching this pattern group: 6620
             work_id work_source                             artist  \
5555048      2970802      master        The Rockin' Home Blues Band   
3408991  10019880044   synthetic  The Starshine Orchestra & Singers   
184809   10036851542   synthetic                       Eric Clapton   
2039097      3709152      master                          Starz (2)   
96496    10001561328   synthetic                               Blur   
209729       3216064      master                        The 45 King   
2663782  10029824306   synthetic                       Binalog Freq   
4156468      1939503      master                             داریوش   
4118664  10032338245   synthetic                         Papash (2)   
473102   10027102141   synthetic                             Module   
2472557      3455633      master                     Amir Alexander   
2408930  10005740401   synthetic                     Harold Betters   
177575       1185730      master     

In [88]:
# Remove the "clear compilation" title group (best of, anthology, box set,
# discography, b-sides, rarities), same reasoning as Greatest Hits and
# Live: a retrospective collection of previously released material isn't
# a new step in the artist's own stylistic trajectory. The 20-row sample
# confirmed this is overwhelmingly real compilations, with one accepted
# edge case ("Best Of Me", which could be an original song title rather
# than a compilation, the same order of noise already accepted for Live
# and reissue titles).
clear_compilation_patterns = r"best of|anthology|box set|discography|b[\s-]?sides|rarities"

is_clear_compilation = works_df["title"].str.contains(
    clear_compilation_patterns, case=False, na=False, regex=True
)

before = len(works_df)
works_df = works_df[~is_clear_compilation].reset_index(drop=True)
after = len(works_df)

print("clear-compilation-titled works removed:", before - after)
print("works_df shape:", works_df.shape)

clear-compilation-titled works removed: 6620
works_df shape: (6009186, 14)


In [89]:
# "collection" is a much more generic word than "best of" or "anthology",
# it could mean a retrospective compilation ("The Singles Collection") or
# just be part of an unrelated original title. Sampling before deciding
# anything, same process as every other title pattern so far.
is_collection = works_df["title"].str.contains(r"collection", case=False, na=False, regex=True)
print("total matching 'collection':", is_collection.sum())

sample = works_df[is_collection].sample(20, random_state=1)
print(sample[["work_id", "work_source", "artist", "title", "year"]])

total matching 'collection': 5361
             work_id work_source                    artist  \
435542   10005683003   synthetic            The Stranglers   
1157269  10008017595   synthetic         Sleepy John Estes   
5859105  10038297871   synthetic         David Garner (11)   
1732217  10009506669   synthetic            Chad VanGaalen   
4617687  10013496885   synthetic               Backpullver   
5981750  10036101977   synthetic            Pandoba Bondre   
4406813  10027797484   synthetic           Summer Of Death   
5387368  10024981457   synthetic                     S.Q.F   
5027017  10026639303   synthetic                   90sFlav   
87468    10030770370   synthetic         Somatic Responses   
341856        254286      master            Gloria Estefan   
5584522  10017027721   synthetic  Murice Laurent Orchestre   
3923844  10016978884   synthetic            Mainstreet (7)   
2412071  10007475757   synthetic                Dexter Dub   
4054322  10034201573   synthetic    

5,362 works match "collection" in the title, but the pattern is too noisy to use for exclusion. Two issues: first, the regex has no word boundary, so it also matches "Recollection" and "Recollections" (twice in a 20-row sample). Second, even past that bug, "collection" doesn't reliably signal a retrospective compilation the way "best of" or "anthology" did: several matches ("Intergalactic Fungi Collection", "Collection Of (Related) Work", "Soulgemcollection") read as original, creatively-named album titles, not a repackaging of previously released material. Unlike Live or the earlier compilation group, where the sample confirmed the pattern meant what it looked like almost every time, here roughly a quarter of the sample is either a false match or genuinely ambiguous. The word is too generic to carry a reliable signal on its own.

Decision

Do not exclude works based on "collection" in the title. The false-positive rate is too high. Adding a word boundary would not fix it, because "collection" also appears in original album titles.


In [90]:
# "remixes" is the biggest pattern in the ambiguous group. The risk here
# is specific: a remix can be a genuine creative work by the credited
# artist (their own reinterpretation), or it can be a compilation of
# OTHER producers' remixes of that artist's tracks, which isn't really
# the credited artist's own new stylistic step. Sampling before deciding,
# same process as everything else.
is_remixes = works_df["title"].str.contains(r"remixes|remix album", case=False, na=False, regex=True)
print("total matching 'remixes':", is_remixes.sum())

sample = works_df[is_remixes].sample(20, random_state=1)
print(sample[["work_id", "work_source", "artist", "title", "year"]])

total matching 'remixes': 24916
             work_id work_source               artist  \
322015        857409      master           Inner Void   
3141169  10009386444   synthetic             Kieran J   
109427       1600168      master           Sharam Jey   
4459835  10013626154   synthetic           Holygunner   
1109986  10007089900   synthetic     Willie Evans Jr.   
3268151  10018712108   synthetic               Divkid   
408003         26943      master  Grindvik & Hardcell   
941644   10006993304   synthetic         Love & Light   
1211776       343745      master     Tommy Four Seven   
5642566      2638778      master       PinkPantheress   
1377263  10002754702   synthetic        Machito Ponce   
825645   10010591169   synthetic       Hector Fonseca   
4637955  10036153583   synthetic           Quark (16)   
202341   10008351461   synthetic          Thunderpuss   
3728713  10022810730   synthetic              Tulioxi   
366928   10019503250   synthetic                 Loft   

In [91]:
# "Remixes" titles are packages of (usually third-party) remixes of an
# already-existing track, credited back to the original artist. Same
# reasoning as reissue: it's a repackaging of existing material, not a
# new creative step for the credited artist. The 20-row sample was
# uniformly clean, no ambiguous originals like we saw with "collection".
is_remixes = works_df["title"].str.contains(r"remixes|remix album", case=False, na=False, regex=True)

before = len(works_df)
works_df = works_df[~is_remixes].reset_index(drop=True)
after = len(works_df)

print("'remixes'-titled works removed:", before - after)
print("works_df shape:", works_df.shape)

'remixes'-titled works removed: 24916
works_df shape: (5984270, 14)


In [92]:
# "tribute" is ambiguous in a different way than "collection": it could be
# a tribute album (other artists covering someone else's songs, credited
# under a tribute-act or various-artists name) which isn't the credited
# artist's own new work, or it could be an original song/album that is
# simply titled as a tribute to someone ("A Tribute To My Father"), which
# IS original work. Sampling before deciding.
is_tribute = works_df["title"].str.contains(r"\btribute\b", case=False, na=False, regex=True)
print("total matching 'tribute':", is_tribute.sum())

sample = works_df[is_tribute].sample(20, random_state=1)
print(sample[["work_id", "work_source", "artist", "title", "year"]])

total matching 'tribute': 5395
             work_id work_source                       artist  \
5724671  10033480860   synthetic                     Heleiðra   
979250   10003659184   synthetic               Vicious Rumors   
5320776      1593189      master                  Sacred Moai   
235817   10014792173   synthetic                     Pianoman   
1542645      1595314      master  Wally Stott & His Orchestra   
3985894       670014      master                 German Brass   
1803474  10001370953   synthetic   The Liquid Crystal Project   
5825679  10034545310   synthetic                 Itziar Yagüe   
2166771      1497153      master         David Murray Quartet   
981751        275598      master                   Iced Earth   
1393980       842409      master                 Hammer Mouth   
526530       1218953      master                  Silence (2)   
4290266  10006145718   synthetic     John Critchinson Quartet   
4965573  10022384636   synthetic           Dennis Stroughma

In [93]:
# "Tribute To X" overwhelmingly means covering/performing someone else's
# material as homage, not the credited artist's own new work, the same
# reasoning as Remixes and the earlier compilation group. A handful of
# standalone matches ("Tribute", "My Tribute") don't name a target and
# could be original compositions, accepted as the same order of noise
# seen in every pattern so far.
is_tribute = works_df["title"].str.contains(r"\btribute\b", case=False, na=False, regex=True)

before = len(works_df)
works_df = works_df[~is_tribute].reset_index(drop=True)
after = len(works_df)

print("'tribute'-titled works removed:", before - after)
print("works_df shape:", works_df.shape)

'tribute'-titled works removed: 5395
works_df shape: (5978875, 14)


In [94]:
# "acoustic" could mean a re-performance of an already-released song in a
# stripped-down arrangement (not new creative content, same logic as Live),
# or it could be an artist's own original acoustic-genre work (where
# "acoustic" is a style descriptor, not a flag that this is a redo of
# something else). Sampling before deciding.
is_acoustic = works_df["title"].str.contains(r"\bacoustic\b", case=False, na=False, regex=True)
print("total matching 'acoustic':", is_acoustic.sum())

sample = works_df[is_acoustic].sample(20, random_state=1)
print(sample[["work_id", "work_source", "artist", "title", "year"]])

total matching 'acoustic': 3824
             work_id work_source                      artist  \
5425627  10014939969   synthetic              The Silent War   
4798315  10019500979   synthetic              Aurora D'Amico   
3819499  10037250793   synthetic                 Paul Jarret   
61346        3913297      master                        Fink   
4978279  10029106385   synthetic                  Jada Facer   
2378820  10008519974   synthetic                 Prago Union   
3492220  10007513806   synthetic                       Debbi   
800449       2271967      master                 Bob Mintzer   
3850388  10009966451   synthetic                John De Jong   
5392823      1660431      master      Ned's Acoustic Dustbin   
4492083  10028979836   synthetic               Landon Austin   
2896799      2033269      master                 Marty Grosz   
2829453  10008083433   synthetic  The Natural History Museum   
4492241  10029044240   synthetic               Landon Austin   
1290032 

In [95]:
# Narrow to the specific pattern "(Acoustic)" as a parenthetical suffix,
# which is what distinguished the real redo cases ("Mantra (Acoustic)")
# from genuine original album titles ("Acoustic Soul", which has no
# parentheses). This should drop the ambiguous standalone matches.
is_acoustic_paren = works_df["title"].str.contains(r"\(acoustic\)", case=False, na=False, regex=True)
print("total matching '(Acoustic)':", is_acoustic_paren.sum())

sample = works_df[is_acoustic_paren].sample(20, random_state=1)
print(sample[["work_id", "work_source", "artist", "title", "year"]])

total matching '(Acoustic)': 1060
             work_id work_source                  artist  \
2533512  10027128493   synthetic                  Neriah   
5108300  10036275530   synthetic        David Keenan (3)   
5418112  10025186564   synthetic           Emily Hackett   
5471523  10030612508   synthetic          Könül Kərimova   
4978349  10029184796   synthetic              Jada Facer   
4119831  10029596414   synthetic                Sam Tsui   
4778518  10033378014   synthetic             Paul Spring   
5444681  10028838626   synthetic             Emily Rowed   
4811530  10015477018   synthetic           Barbara Pravi   
1297936      2431990      master             Silverstein   
2017877  10016183123   synthetic              Kevin Simm   
5102838  10030495400   synthetic  Charlotte Lawrence (2)   
634242   10026342651   synthetic           Cherie Currie   
4940332  10016042273   synthetic            Randall King   
3209954  10003493769   synthetic     Cheerleaders United   
539178

In [96]:
# Narrowed to the "(Acoustic)" parenthetical pattern, which the 20-row
# sample confirmed is uniformly a redo of an already-existing song, not
# an original album title. Same reasoning as Live: a re-performance of
# existing material isn't a new creative work.
is_acoustic_paren = works_df["title"].str.contains(r"\(acoustic\)", case=False, na=False, regex=True)

before = len(works_df)
works_df = works_df[~is_acoustic_paren].reset_index(drop=True)
after = len(works_df)

print("'(Acoustic)'-titled works removed:", before - after)
print("works_df shape:", works_df.shape)

'(Acoustic)'-titled works removed: 1060
works_df shape: (5977815, 14)


In [97]:
# Same approach as acoustic: try the narrow parenthetical pattern first,
# "(Instrumental)" as a suffix on an existing song title, rather than the
# word anywhere in the title (which would also catch genuinely original
# instrumental music as a genre/style, same risk as "Acoustic Soul").
is_instrumental_paren = works_df["title"].str.contains(r"\(instrumental\)", case=False, na=False, regex=True)
print("total matching '(Instrumental)':", is_instrumental_paren.sum())

sample = works_df[is_instrumental_paren].sample(min(20, is_instrumental_paren.sum()), random_state=1)
print(sample[["work_id", "work_source", "artist", "title", "year"]])

total matching '(Instrumental)': 757
             work_id work_source                   artist  \
5695544  10025114411   synthetic            Alex Dionisio   
5491258  10025394011   synthetic                 Namo (6)   
5530785  10016243539   synthetic               Calee Reed   
5887231  10031569010   synthetic              Archèometre   
4706165  10008660529   synthetic                 Safehaus   
1504794      4078318      master           Timothy Wilson   
3501842  10011317517   synthetic                CRAVE (6)   
3506657  10016006979   synthetic      Jesus Loves Junkies   
5473566  10018217693   synthetic         Tony Martin (47)   
4163167      1536986      master           Stoneheart (2)   
3724317  10038194092   synthetic              Dj Flux (4)   
5948310  10038229354   synthetic      Debajit Raikarmakar   
3813116  10008177449   synthetic             Yoshie Saito   
107637   10031129328   synthetic             Simon Harris   
337959        137930      master              Is

In [98]:
# Same narrow pattern as acoustic: "(Instrumental)" as a suffix on an
# existing song title (vocal-less version), not the word anywhere in the
# title. Sample of 20 confirmed the pattern, same reasoning as Live and
# the acoustic-paren pattern: not a new creative step.
is_instrumental_paren = works_df["title"].str.contains(r"\(instrumental\)", case=False, na=False, regex=True)

before = len(works_df)
works_df = works_df[~is_instrumental_paren].reset_index(drop=True)
after = len(works_df)

print("'(Instrumental)'-titled works removed:", before - after)
print("works_df shape:", works_df.shape)

'(Instrumental)'-titled works removed: 757
works_df shape: (5977058, 14)


In [99]:
# "complete" is as generic a word as "collection", risk is the same:
# "The Complete Recordings" is a compilation, but "complete" could also
# appear in an unrelated original title. Sampling before deciding.
is_complete = works_df["title"].str.contains(r"\bcomplete\b", case=False, na=False, regex=True)
print("total matching 'complete':", is_complete.sum())

sample = works_df[is_complete].sample(20, random_state=1)
print(sample[["work_id", "work_source", "artist", "title", "year"]])

total matching 'complete': 1614
             work_id work_source                              artist  \
457146   10016299569   synthetic                  The Gerogerigegege   
2357227       787782      master                           Iscariote   
292947   10007434011   synthetic                     Louis Armstrong   
437306       3616105      master                              Hanson   
2819468  10002671205   synthetic                       Guy Gross (2)   
1488343  10010040104   synthetic                       Frank Loesser   
4722446  10036831625   synthetic                         Esmectatons   
2671890  10035864614   synthetic                        Fred Sokolow   
3948937  10008079943   synthetic                              Danglo   
444799   10009651821   synthetic                      Charlie Parker   
780108   10010228639   synthetic                      Giuseppe Verdi   
4756831      3091671      master                       Akasha System   
1669530  10004192862   synthetic

In [100]:
# "unplugged" is more specific than "acoustic"/"complete", usually refers
# to an MTV Unplugged-style live acoustic performance of existing songs.
# Sampling before deciding, same process as everything else.
is_unplugged = works_df["title"].str.contains(r"unplugged", case=False, na=False, regex=True)
print("total matching 'unplugged':", is_unplugged.sum())

sample = works_df[is_unplugged].sample(20, random_state=1)
print(sample[["work_id", "work_source", "artist", "title", "year"]])

total matching 'unplugged': 983
             work_id work_source                          artist  \
3903470  10014469155   synthetic  Interdimensional Vortex League   
3706427  10013196718   synthetic                      El Jeffrey   
4232375  10022553732   synthetic                  Θοδωρής Φέρρης   
2198896       273912      master                      Katy Perry   
3928341      1849980      master                             Ba.   
556373        108686      master                       The Corrs   
492084       1218048      master                  Violent Femmes   
2130651  10019088899   synthetic                  Najoua Belyzel   
398588   10025227064   synthetic                       Scorpions   
3077623  10011761375   synthetic                Wilfran Castillo   
5439998  10022283653   synthetic                    Deejay Young   
73847    10027040938   synthetic                   Terry Callier   
3717282  10037076118   synthetic                    Murkage Dave   
2502884      220

In [101]:
# "unplugged" overwhelmingly means an MTV-Unplugged-style live acoustic
# performance of already-existing songs, same reasoning as Live. Sample
# of 20 was uniformly clean.
is_unplugged = works_df["title"].str.contains(r"unplugged", case=False, na=False, regex=True)

before = len(works_df)
works_df = works_df[~is_unplugged].reset_index(drop=True)
after = len(works_df)

print("'unplugged'-titled works removed:", before - after)
print("works_df shape:", works_df.shape)

'unplugged'-titled works removed: 983
works_df shape: (5976075, 14)


In [102]:
# "karaoke" is specific and unlikely to appear in a genuine original
# title, usually a backing-track/sing-along version aimed at karaoke use,
# not the artist's own new creative work. Sampling before deciding.
is_karaoke = works_df["title"].str.contains(r"karaoke", case=False, na=False, regex=True)
print("total matching 'karaoke':", is_karaoke.sum())

sample = works_df[is_karaoke].sample(20, random_state=1)
print(sample[["work_id", "work_source", "artist", "title", "year"]])

total matching 'karaoke': 426
             work_id work_source                              artist  \
1912899  10036594175   synthetic               The Countdown Singers   
1028375  10000761460   synthetic                           Blanks 77   
1476771  10036063493   synthetic                         Frans Bauer   
5098316  10015310232   synthetic  WZ-Orkiestra & Зьмiцер Вайцюшкевiч   
1675589  10005172775   synthetic                    Paul Dempsey (2)   
4578371  10035676451   synthetic                             Tuấn Vũ   
3920052  10021205585   synthetic                                 ぐりむ   
27909          60063      master                           Omni Trio   
3645089  10005199499   synthetic                      Electric Magma   
1448056      3886537      master                  The Automatics (2)   
5001553  10034457305   synthetic                         Roel Cortez   
64081          44333      master                       Takkyu Ishino   
605658   10016003554   synthetic  

In [103]:
# Quick volume check on the remaining title candidates before deciding
# anything: BBC/Peel-style radio sessions (a live re-performance of
# existing material, same concept as Unplugged) and radio edits/versions
# (a shortened derivative of an existing track, not new work).
candidate_patterns = {
    "session": r"\bsession(s)?\b",
    "bbc session": r"bbc session",
    "radio edit": r"radio edit",
    "radio version": r"radio version",
}

for label, pattern in candidate_patterns.items():
    count = works_df["title"].str.contains(pattern, case=False, na=False, regex=True).sum()
    print(f"{label}: {count}")

/var/folders/b4/svxttqvs5zg_rg_drrxn_vf00000gn/T/ipykernel_10487/703692964.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  count = works_df["title"].str.contains(pattern, case=False, na=False, regex=True).sum()


session: 13418
bbc session: 104


radio edit: 1468
radio version: 166


In [104]:
# "session" alone is suspected to be too generic (e.g. "Jam Session" as an
# original album title). "radio edit" is a more specific, likely cleaner
# pattern. Sampling both before deciding.
is_session = works_df["title"].str.contains(r"\bsession(s)?\b", case=False, na=False, regex=True)
print("SESSION sample:")
print(works_df[is_session].sample(15, random_state=1)[["work_id", "artist", "title", "year"]])

print()
is_radio_edit = works_df["title"].str.contains(r"radio edit", case=False, na=False, regex=True)
print("RADIO EDIT sample:")
print(works_df[is_radio_edit].sample(15, random_state=1)[["work_id", "artist", "title", "year"]])

/var/folders/b4/svxttqvs5zg_rg_drrxn_vf00000gn/T/ipykernel_10487/2745550481.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_session = works_df["title"].str.contains(r"\bsession(s)?\b", case=False, na=False, regex=True)


SESSION sample:
             work_id                  artist  \
5031537  10020278987  The Redemption's Colts   
2783841       916705         Bing Satellites   
4467093  10019102033              Last Train   
1753810  10023771558               Superdrag   
3272640      1961743             Helicon (4)   
138713        141714         Sarah McLachlan   
671122         40674       James Blood Ulmer   
5069058  10032108058                  Calous   
4682448  10022672781     Caleb R.K. Williams   
4306964  10034373809                    Jato   
2348469  10009362055          Passenger (10)   
2530876  10034259470           Jasper Erkens   
1983446  10015313030            Joshua James   
5415719      1541650            Pyrotechnist   
5376959  10014427061        Yoshi Suzuki (2)   

                                                    title    year  
5031537                             The basement sessions  2020.0  
2783841                       Twilight Sessions Volume 17  2015.0  
4467093    

RADIO EDIT sample:
             work_id                    artist  \
4350613  10018100195                Bonny Doon   
2262341  10038166546            Dierks Bentley   
2112532      1355737             Orquestra Was   
3853259  10009662938  La Torre Dell'Alchimista   
3457759  10038267451                      Memz   
5222892  10017800465        Loose Canon Playaz   
5155258  10015717522            Left At London   
1917672  10012217215             Marlene Kuntz   
1750528  10005004194            Karine Polwart   
1232160  10015087857                  !Deladap   
5646756  10025413742            Pale Mannequin   
5682267  10022085362                 DJ Clipps   
4856101  10009693821             Hungry Bodies   
137059   10009666187            Monster Magnet   
5561983  10022386478                Die Lauser   

                                                     title    year  
4350613                           Lost My Way (Radio Edit)  2017.0  
2262341                             I Hold

In [105]:
# "session" alone was too generic (mostly original recording-session
# albums, not re-performances), discarded. "radio edit"/"radio version"
# are a clean, specific pattern: a shortened derivative of an existing
# track, not new work. "bbc session" is the same concept as Unplugged,
# small but unambiguous.
is_radio_derivative = works_df["title"].str.contains(
    r"radio edit|radio version|radio editon|bbc session", case=False, na=False, regex=True
)

before = len(works_df)
works_df = works_df[~is_radio_derivative].reset_index(drop=True)
after = len(works_df)

print("radio-edit/version/bbc-session works removed:", before - after)
print("works_df shape:", works_df.shape)

radio-edit/version/bbc-session works removed: 1735
works_df shape: (5974340, 14)


### Question: Are there any noise in artist name?

In [106]:
# Revisit the pending QA item: how many placeholder-like artist names
# (Various, Unknown Artist, No Artist, and close variants in casing,
# spacing, or wording) are still in works_df after the original exact
# match exclusion? This checks both the known leak and whether there are
# other spellings the original filter missed.
placeholder_pattern = r"^various( artists?)?$|^unknown( artist)?$|^no artist$|^n/?a$|^v\.?a\.?$"
is_placeholder_leak = works_df["artist"].str.contains(
    placeholder_pattern, case=False, na=False, regex=True
)
print("placeholder-like artist names still present:", is_placeholder_leak.sum())
print(works_df[is_placeholder_leak][["work_id", "work_source", "artist", "title", "year"]].head(20))

placeholder-like artist names still present: 1422
          work_id work_source   artist  \
4517  10031589146   synthetic  Various   
4518  10017163802   synthetic  Various   
4519  10029834638   synthetic  Various   
4520  10009383656   synthetic  Various   
4521  10013945672   synthetic  Various   
4522  10035727529   synthetic  Various   
4523      1435914      master  Various   
4524  10005242034   synthetic  Various   
4525  10035620984   synthetic  Various   
4526  10028431436   synthetic  Various   
4527  10036133384   synthetic  Various   
4528  10031656485   synthetic  Various   
4529  10007831849   synthetic  Various   
4530  10011518053   synthetic  Various   
4531  10000945974   synthetic  Various   
4532      1394129      master  Various   
4533  10002871653   synthetic  Various   
4534      4002532      master  Various   
4535  10010873444   synthetic  Various   
4536  10011739526   synthetic  Various   

                                                  title    year  
4

/var/folders/b4/svxttqvs5zg_rg_drrxn_vf00000gn/T/ipykernel_10487/3659690898.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_placeholder_leak = works_df["artist"].str.contains(


In [107]:
# Same root cause as the year==0 bug: the original placeholder-artist
# exclusion did an exact, case-sensitive string match, so "various",
# "VARIOUS" etc. slipped through alongside the correctly excluded
# "Various". This isn't ambiguous content, it's a case-sensitivity gap
# in the earlier filter, same fix pattern as before.
placeholder_pattern = r"^various( artists?)?$|^unknown( artist)?$|^no artist$|^n/?a$|^v\.?a\.?$"
is_placeholder_leak = works_df["artist"].str.contains(
    placeholder_pattern, case=False, na=False, regex=True
)

before = len(works_df)
works_df = works_df[~is_placeholder_leak].reset_index(drop=True)
after = len(works_df)

print("placeholder-artist works removed:", before - after)
print("works_df shape:", works_df.shape)

/var/folders/b4/svxttqvs5zg_rg_drrxn_vf00000gn/T/ipykernel_10487/3229533640.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_placeholder_leak = works_df["artist"].str.contains(


placeholder-artist works removed: 1422
works_df shape: (5972918, 14)


In [108]:
# Check 1: empty or whitespace-only artist names.
# Check 2: same artist name (normalized: lowercase, stripped) spread across
# more than one artist_id, WITHOUT a Discogs disambiguation suffix like
# "(2)" in the name itself. A legitimate disambiguated name already looks
# different ("Queen (2)" vs "Queen"), so this only catches cases where the
# same literal name got split into separate artist_ids by accident
# (casing, stray whitespace), not Discogs' own intentional disambiguation.
empty_artist = works_df["artist"].str.strip().eq("")
print("empty/whitespace-only artist names:", empty_artist.sum())

normalized_artist = works_df["artist"].str.strip().str.lower()
artist_id_per_name = works_df.groupby(normalized_artist)["artist_id"].nunique()
split_names = artist_id_per_name[artist_id_per_name > 1]
print("normalized artist names spread across multiple artist_ids:", len(split_names))
print(split_names.sort_values(ascending=False).head(15))

empty/whitespace-only artist names: 0


normalized artist names spread across multiple artist_ids: 21
artist
arlhow                        2
yør                           2
ненависть                     2
лихолесье                     2
космическая пыль              2
забытая память                2
глина!!! глина!!! глина!!!    2
бетон                         2
ölüm                          2
zoë                           2
vääristymä                    2
høst                          2
salò                          2
rán                           2
nüur                          2
Name: artist_id, dtype: int64


In [109]:
import unicodedata

# Confirm the Unicode-normalization hypothesis: if the two artist_ids for
# "zoë" are really the same name encoded two different ways (precomposed
# "ë" vs. "e" + combining diaeresis), NFC-normalizing both should collapse
# them to an identical string.
examples = ["zoë", "høst", "yør"]
for name in examples:
    rows = works_df[works_df["artist"].str.strip().str.lower() == name]
    for raw in rows["artist"].unique():
        print(repr(name), "->", repr(raw), "NFC:", repr(unicodedata.normalize("NFC", raw)))

'zoë' -> 'Zoë' NFC: 'Zoë'
'zoë' -> 'ZOË' NFC: 'ZOË'


'høst' -> 'Høst' NFC: 'Høst'
'høst' -> 'HØST' NFC: 'HØST'


'yør' -> 'Yør' NFC: 'Yør'
'yør' -> 'YØR' NFC: 'YØR'


These are different artists, so no action is taken.

### Question: Did the title-based dedup in 01_cleaning_eda.ipynb actually catch all near-duplicates, now that some artists show very high work counts?

In [110]:
from eda import inspect_near_duplicate_titles  # reusable fuzzy-matching function lives in src/eda.py

inspect_near_duplicate_titles(works_df, "Bing Crosby")  # Bing Crosby's titles ranked by similarity to their closest match

,work_id,title,year,work_source,best_match_score,best_match_title
1139,10038289771,Sweet Leilani / In A Little Hula Heaven,NaN,synthetic,100.0,Sweet Leilani / In A Little Hula Heaven
118,10035748076,Beautiful Girl / The Day You Came Along,NaN,synthetic,100.0,The Day You Came Along / Beautiful Girl
500,10014813278,Holiday In Europe,1963.0,synthetic,100.0,Holiday In Europe
499,554051,Holiday In Europe,1962.0,master,100.0,Holiday In Europe
496,1547369,High Society,NaN,master,100.0,High Society
495,10005482991,High Society,1956.0,synthetic,100.0,High Society
463,10024064922,Hang Your Heart On A Hickory Limb / That Sly O...,1940.0,synthetic,100.0,That Sly Old Gentleman / Hang Your Heart On A ...
459,10016535298,Granada,1953.0,synthetic,100.0,Granada
458,10034392895,Granada,NaN,synthetic,100.0,Granada
438,10013167071,Going My Way,1953.0,synthetic,100.0,Going My Way


### Interpretation

Two different patterns show up in Bing Crosby's top near-duplicate matches:

- **Same-title, different-year clusters** ("White Christmas" x6 across 1970-2010,
  "Bing Sings Whilst Bregman Swings" x3, "Blue Skies" x3, "Going My Way",
  "Granada", "High Society", "Don't Fence Me In" x2 each): these are genuinely
  ambiguous from title/year alone. They may be legitimate distinct releases (a
  signature song re-pressed as new singles/compilations across decades, common for
  an artist this prolific) that the existing reissue filter correctly left alone
  because they weren't tagged reissue/repress/remaster, or they may be true
  duplicates that filter missed. Title and year alone can't resolve this; it would
  need format_descriptions or a release-level look.
- **Minor-wording near-duplicates** ("Drifting And Dreaming" vs "Drifting Dreaming",
  "East Side Of Heaven" vs "East Side Heaven", "Where Blue Night" vs "Where Blue Of
  Night"): these are not ambiguous. Each pair is almost certainly the same work,
  entered with a dropped word, something the original exact-match `title_normalized`
  dedup could never catch since the normalized strings genuinely differ.

This is a real, if narrow, gap: the existing dedup only catches exact-after-normalization
title matches, not near-duplicates a few words apart.

### Decision

No filtering applied here (EDA-proper stage, not cleaning). Logged as a new PENDING
item in `docs/decisions_core.md`: whether to add a fuzzy-match dedup pass (e.g. rapidfuzz
token_sort_ratio above some threshold, same artist_id) on top of the existing
exact-match dedup during cleaning, and how to resolve the separate same-title/
different-year ambiguity (needs format_descriptions, not resolvable from title/year
alone).

### Question: Before writing a fuzzy-dedup filter, what similarity score reliably separates true duplicates from genuinely distinct titles?

In [111]:
from eda import sample_pairs_by_similarity_band  # diagnostic sampler lives in src/eda.py

# sample candidate pairs by similarity band, within the same artist_id, with the number/roman-numeral guard enabled
similarity_band_samples = sample_pairs_by_similarity_band(works_df, guard_number_or_roman=True)

for band, pairs in similarity_band_samples.items():  # one block per requested similarity band
    print(f"--- {band} ---")  # band label as a visual separator
    print(pairs[["score", "title_a", "title_b", "year_a", "year_b", "work_source_a", "work_source_b"]].to_string(index=False))  # the sampled pairs for manual inspection
    print()  # blank line between bands


Comparing titles within 695424 artists (5633490 works), window size 10


--- 85-90 ---
    score                                                                     title_a                                                                    title_b  year_a  year_b work_source_a work_source_b
86.363636                                                         Orquesta Reciclando                                                  Orquesta Reciclando Fotos  2009.0  2009.0        master     synthetic
89.552239                                              Donato Racciatti Y Su Orquesta                                      Donato Racciatti Y Su Orquesta Tipica  1959.0     NaN     synthetic     synthetic
89.932886 Avatar: The Last Airbender - Book 1: Water (Music From The Animated Series) Avatar: The Last Airbender – Book 3: Fire (Music From The Animated Series)  2023.0  2026.0        master     synthetic
85.714286                                                     Oklahoma!  (Part Three)                                                       Oklahoma! (Part Two)  1959

### Interpretation

With the number/roman-numeral guard applied (including stripping digits fused onto a word, so "mix4"/"mix7" and "S01E03"/"S01E05" are excluded the same way as "Vol. 2"/"Vol. 3"), the 85-90 and 90-95 bands still carry some noise (shared medley/compilation templates, EP suffixes, volume-series pairs using "Vol."/"Volume" spelled out differently), but the 95-99 band is almost entirely genuine near-duplicates: typos, accent variants, punctuation differences, and "and"/"&" swaps. A threshold around 90 captures most real duplicates while keeping the guard's false-positive rate low; going as high as 95 would be cleaner but would miss known true duplicates in Bing Crosby's catalog (scores 89.5 and 91.4).


### Decision

Use score_threshold=90 with the guard (including fused-digit stripping) for the fuzzy-dedup filter. Within each artist_id, collapse any pair scoring 90 or higher on token_sort_ratio and passing the guard, keeping the earlier-year work (missing year treated as latest, ties broken by smaller work_id). Two known limitations are accepted rather than adding further normalization: (1) "Vol."/"Volume" abbreviation mismatches aren't caught by the guard; (2) shared-template titles with a non-numeric varying word (e.g. a medley "X / Y" title where Y differs) can still produce a false positive. Implemented as `remove_fuzzy_duplicate_titles` in `src/works.py`, applied here at the end of the pipeline, right before the final checkpoint.


In [112]:
from works import remove_fuzzy_duplicate_titles  # production fuzzy-dedup filter lives in src/works.py

before_shape = works_df.shape  # record the row count before collapsing near-duplicate titles
works_df = remove_fuzzy_duplicate_titles(works_df)  # collapse wording-level near-duplicate titles within the same artist_id
after_shape = works_df.shape  # record the row count after the filter

print("before:", before_shape)  # rows before the fuzzy-dedup filter
print("after:", after_shape)  # rows after the fuzzy-dedup filter
print("rows removed:", before_shape[0] - after_shape[0])  # total rows collapsed as near-duplicates


before: (5972918, 14)
after: (5952053, 14)
rows removed: 20865


#### Checkpoint: post fuzzy-dedup

In [113]:
# Drop the helper columns used mid-cleaning that are now genuinely dead
# weight: has_earlier_same_title and is_reissue_candidate only mattered for
# the reissue-candidate filter (already applied, those rows are gone), and
# is_live_tagged was always False (format_descriptions never carried a
# Live value, title-matching replaced it). title_normalized and
# is_reissue_tagged are kept alive, the catalog-number dedup filter below
# still needs both.
works_df = works_df.drop(columns=[
    "has_earlier_same_title", "is_reissue_candidate", "is_live_tagged",
])

works_df.to_parquet("../data/processed/works_checkpoint_fuzzy_dedup.parquet", index=False)
print("works_df shape:", works_df.shape)

works_df shape: (5952053, 11)


#### Catalog-number duplicate collapse

Within the 55,642 untagged, same-artist/same-title, multi-year groups (122,569 rows) that the existing reissue-tag filter has no visibility into, 951 groups (1,918 rows) share an exact label name + real catalog number between two or more entries of different year/format. A 30-row sample at this threshold was uniformly clean (same release, a later synthetic entry or a reissue Discogs never tagged). A further 2,943 groups share only a blank/"none" catalog number, excluded as weak evidence: two unrelated self-releases can easily both be uncatalogued.

Collapse the 951 matched groups using the same tie-break as the fuzzy-dedup filter: keep the earliest year (a missing year sorts last), ties broken by the smaller work_id. Implemented as `remove_catalog_number_duplicates` in `src/works.py`.

In [114]:
# Reload releases_df with only the columns this filter needs: label/catno
# info (not kept earlier) plus the three columns needed to pick a
# representative release per master (is_main_release, released) and to
# link a synthetic Work back to its own release (release_id).
RELEASE_COLUMNS_FOR_CATALOG = [
    "release_id", "master_id", "is_main_release", "released", "labels",
]
releases_df = pd.read_parquet("../data/processed/releases.parquet", columns=RELEASE_COLUMNS_FOR_CATALOG)

from works import remove_catalog_number_duplicates  # production catalog-number dedup filter lives in src/works.py

before_shape = works_df.shape  # record the row count before collapsing catalog-number matches
works_df = remove_catalog_number_duplicates(works_df, releases_df)  # collapse same label+catno matches within untagged multi-year groups
after_shape = works_df.shape  # record the row count after the filter

del releases_df  # drop the 19.4M-row releases table from memory, not needed again

print("before:", before_shape)  # rows before the catalog-number dedup filter
print("after:", after_shape)  # rows after the catalog-number dedup filter
print("rows removed:", before_shape[0] - after_shape[0])  # total rows collapsed as catalog-number matches

before: (5952053, 11)
after: (5951071, 11)
rows removed: 982


#### Mass-upload artist exclusion

"Nosferatu 1922" (artist_id 6391827, 1,854 works) surfaced in EDA as a volume outlier whose catalog reads as a bulk/algorithmic upload, not a discography: almost all works synthetic (standalone releases, no Discogs Master), almost all titles bare roman numerals, and almost all landing inside a single two-year window. None of four genuinely prolific artists checked (Bing Crosby, Elvis Presley, Frank Sinatra, Duke Ellington) show more than one of these signals at once.

Rule, applied only to artists with 50+ works (the floor needed so the other two signals mean anything; it sits at roughly the 99th percentile of works-per-artist): synthetic_share >= 0.90 AND roman_numeral_title_share >= 0.50 AND busiest_2yr_window_share >= 0.80, checked against every artist_id in works_df, not a fixed exclusion list. Implemented as `remove_mass_upload_artists` in `src/works.py`. Without the 50-work floor, the same rule matches 165 artists / 2,301 rows, but 164 of those have only 2-18 works, too few for the volume/window signals to carry any real meaning.

In [115]:
from works import remove_mass_upload_artists  # production mass-upload exclusion filter lives in src/works.py

before_shape = works_df.shape  # record the row count before excluding mass-upload artists
works_df = remove_mass_upload_artists(works_df)  # drop artist_ids matching the mass-upload signature (50+ works floor applied inside)
after_shape = works_df.shape  # record the row count after the filter

print("before:", before_shape)  # rows before the mass-upload exclusion filter
print("after:", after_shape)  # rows after the mass-upload exclusion filter
print("rows removed:", before_shape[0] - after_shape[0])  # total rows removed as mass-upload artists

before: (5951071, 11)
after: (5949217, 11)
rows removed: 1854


#### Checkpoint: final

In [116]:
# title_normalized and is_reissue_tagged were only needed by the catalog-
# number dedup filter just above; both are dead weight now that it has run.
# This is the true end of the cleaning pipeline: works_checkpoint.parquet is
# reserved exclusively for this final state, any cleaning step added later
# must be inserted before this cell, never after it.
works_df = works_df.drop(columns=["title_normalized", "is_reissue_tagged"])

works_df.to_parquet("../data/processed/works_checkpoint.parquet", index=False)
print("works_df shape:", works_df.shape)

works_df shape: (5949217, 9)
